In [ ]:
import zipfile
import os
import json
import glob

zip_path = "/content/tests_male_json.zip"
extract_path = "/content/test_data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

json_files = glob.glob(
    extract_path + "/**/*.json",
    recursive=True
)

print("Total Test JSON files:", len(json_files))

Total Test JSON files: 894


In [ ]:
with open(json_files[0], "r", encoding="utf-8") as f:
    match = json.load(f)

print("Top-level keys:", match.keys())

info = match["info"]

print("Teams:", info.get("teams"))
print("Date:", info.get("dates"))
print("Venue:", info.get("venue"))
print("Match Type:", info.get("match_type"))
print("Gender:", info.get("gender"))
print("Outcome:", info.get("outcome"))

Top-level keys: dict_keys(['meta', 'info', 'innings'])
Teams: ['England', 'Australia']
Date: ['2009-07-30', '2009-07-31', '2009-08-01', '2009-08-02', '2009-08-03']
Venue: Edgbaston
Match Type: Test
Gender: male
Outcome: {'result': 'draw'}


In [ ]:
import pandas as pd

matches = []

for file in json_files:
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        info = data["info"]
        teams = info.get("teams", [])
        outcome = info.get("outcome", {})

        if len(teams) != 2:
            continue

        if "winner" not in outcome:
            continue

        winner = outcome["winner"]

        if winner not in teams:
            continue

        matches.append({
            "match_id": os.path.splitext(
                os.path.basename(file)
            )[0],

            "date": str(
                info.get("dates", [""])[0]
            ),

            "season": info.get("season"),

            "venue": info.get(
                "venue",
                "Unknown"
            ),

            "city": info.get(
                "city",
                "Unknown"
            ),

            "team_1": teams[0],
            "team_2": teams[1],

            "winner": winner,

            "match_type": info.get(
                "match_type"
            ),

            "gender": info.get(
                "gender"
            ),

            "event": info.get(
                "event", {}
            ).get(
                "name",
                "Unknown"
            ),

            "toss_winner": info.get(
                "toss", {}
            ).get(
                "winner",
                "Unknown"
            ),

            "toss_decision": info.get(
                "toss", {}
            ).get(
                "decision",
                "Unknown"
            )
        })

    except Exception:
        continue

test_matches = pd.DataFrame(matches)

test_matches["date"] = pd.to_datetime(
    test_matches["date"],
    errors="coerce"
)

test_matches = (
    test_matches
    .sort_values("date")
    .reset_index(drop=True)
)

print("Valid Test matches:", len(test_matches))
test_matches.head()

Valid Test matches: 725


,match_id,date,season,venue,city,team_1,team_2,winner,match_type,gender,event,toss_winner,toss_decision
0,64022,2002-12-26,2002/03,"Kingsmead, Durban",Durban,South Africa,Pakistan,South Africa,Test,male,Pakistan tour of South Africa,Pakistan,field
1,64012,2002-12-26,2002/03,Melbourne Cricket Ground,Melbourne,Australia,England,Australia,Test,male,England tour of Australia,Australia,bat
2,64013,2003-01-02,2002/03,Sydney Cricket Ground,Sydney,England,Australia,England,Test,male,England tour of Australia,England,bat
3,64023,2003-01-02,2002/03,"Newlands, Cape Town",Cape Town,South Africa,Pakistan,South Africa,Test,male,Pakistan tour of South Africa,South Africa,bat
4,64027,2003-05-09,2003,"Antigua Recreation Ground, St John's, Antigua",St John's,Australia,West Indies,West Indies,Test,male,Australia tour of West Indies,Australia,bat


In [ ]:
team_performance = []

for file in json_files:
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        info = data["info"]
        teams = info.get("teams", [])

        if len(teams) != 2:
            continue

        match_id = os.path.splitext(
            os.path.basename(file)
        )[0]

        for innings in data.get("innings", []):

            team = innings.get("team")

            if team not in teams:
                continue

            total_runs = 0
            wickets = 0
            legal_balls = 0

            for over in innings.get("overs", []):

                for delivery in over.get(
                    "deliveries", []
                ):

                    runs = delivery.get(
                        "runs", {}
                    )

                    total_runs += runs.get(
                        "total", 0
                    )

                    extras = delivery.get(
                        "extras", {}
                    )

                    if (
                        "wides" not in extras
                        and "noballs" not in extras
                    ):
                        legal_balls += 1

                    for wicket in delivery.get(
                        "wickets", []
                    ):

                        dismissal = wicket.get(
                            "kind",
                            ""
                        )

                        if dismissal not in [
                            "retired hurt",
                            "retired not out",
                            "obstructing the field"
                        ]:
                            wickets += 1

            overs = legal_balls / 6

            run_rate = (
                total_runs / overs
                if overs > 0
                else 0
            )

            team_performance.append({
                "match_id": match_id,
                "team": team,
                "runs": total_runs,
                "wickets": wickets,
                "legal_balls": legal_balls,
                "run_rate": run_rate
            })

    except Exception:
        continue

test_performance = pd.DataFrame(
    team_performance
)

print(
    "Performance records:",
    len(test_performance)
)

test_performance.head()

Performance records: 3322


,match_id,team,runs,wickets,legal_balls,run_rate
0,345972,Australia,263,10,424,3.721698
1,345972,England,376,10,561,4.021390
2,345972,Australia,375,5,674,3.338279
3,743965,Australia,566,8,894,3.798658
4,743965,England,312,10,541,3.460259


In [ ]:
team1_perf = test_performance.rename(
    columns={
        "team": "team_1",
        "runs": "team_1_runs",
        "wickets": "team_1_wickets",
        "legal_balls": "team_1_legal_balls",
        "run_rate": "team_1_run_rate"
    }
)

team2_perf = test_performance.rename(
    columns={
        "team": "team_2",
        "runs": "team_2_runs",
        "wickets": "team_2_wickets",
        "legal_balls": "team_2_legal_balls",
        "run_rate": "team_2_run_rate"
    }
)

test_matches = test_matches.merge(
    team1_perf[
        [
            "match_id",
            "team_1",
            "team_1_runs",
            "team_1_wickets",
            "team_1_legal_balls",
            "team_1_run_rate"
        ]
    ],
    on=["match_id", "team_1"],
    how="left"
)

test_matches = test_matches.merge(
    team2_perf[
        [
            "match_id",
            "team_2",
            "team_2_runs",
            "team_2_wickets",
            "team_2_legal_balls",
            "team_2_run_rate"
        ]
    ],
    on=["match_id", "team_2"],
    how="left"
)

print(
    "Dataset shape:",
    test_matches.shape
)

test_matches.head()

Dataset shape: (2562, 21)


,match_id,date,season,venue,city,team_1,team_2,winner,match_type,gender,...,toss_winner,toss_decision,team_1_runs,team_1_wickets,team_1_legal_balls,team_1_run_rate,team_2_runs,team_2_wickets,team_2_legal_balls,team_2_run_rate
0,64022,2002-12-26,2002/03,"Kingsmead, Durban",Durban,South Africa,Pakistan,South Africa,Test,male,...,Pakistan,field,368,10,642,3.439252,161,10,292,3.308219
1,64022,2002-12-26,2002/03,"Kingsmead, Durban",Durban,South Africa,Pakistan,South Africa,Test,male,...,Pakistan,field,368,10,642,3.439252,250,10,525,2.857143
2,64022,2002-12-26,2002/03,"Kingsmead, Durban",Durban,South Africa,Pakistan,South Africa,Test,male,...,Pakistan,field,45,0,54,5.000000,161,10,292,3.308219
3,64022,2002-12-26,2002/03,"Kingsmead, Durban",Durban,South Africa,Pakistan,South Africa,Test,male,...,Pakistan,field,45,0,54,5.000000,250,10,525,2.857143
4,64012,2002-12-26,2002/03,Melbourne Cricket Ground,Melbourne,Australia,England,Australia,Test,male,...,Australia,bat,551,6,877,3.769669,270,10,537,3.016760


In [ ]:
test_matches["target"] = (
    test_matches["winner"]
    == test_matches["team_1"]
).astype(int)

print("Target distribution:")
print(
    test_matches["target"]
    .value_counts()
)

print("\nColumns:")
print(
    test_matches.columns.tolist()
)

Target distribution:
target
1    1496
0    1066
Name: count, dtype: int64

Columns:
['match_id', 'date', 'season', 'venue', 'city', 'team_1', 'team_2', 'winner', 'match_type', 'gender', 'event', 'toss_winner', 'toss_decision', 'team_1_runs', 'team_1_wickets', 'team_1_legal_balls', 'team_1_run_rate', 'team_2_runs', 'team_2_wickets', 'team_2_legal_balls', 'team_2_run_rate', 'target']


In [ ]:
from collections import defaultdict

team_stats = defaultdict(lambda: {
    "matches": 0,
    "wins": 0,
    "runs": [],
    "wickets": [],
    "run_rates": []
})

historical_rows = []

for _, row in test_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    s1 = team_stats[team1]
    s2 = team_stats[team2]

    historical_rows.append({
        "team_1_win_rate": (
            s1["wins"] / s1["matches"]
            if s1["matches"] else 0.5
        ),
        "team_2_win_rate": (
            s2["wins"] / s2["matches"]
            if s2["matches"] else 0.5
        ),
        "team_1_avg_runs": (
            sum(s1["runs"]) / len(s1["runs"])
            if s1["runs"] else 0
        ),
        "team_2_avg_runs": (
            sum(s2["runs"]) / len(s2["runs"])
            if s2["runs"] else 0
        ),
        "team_1_avg_wickets": (
            sum(s1["wickets"]) / len(s1["wickets"])
            if s1["wickets"] else 0
        ),
        "team_2_avg_wickets": (
            sum(s2["wickets"]) / len(s2["wickets"])
            if s2["wickets"] else 0
        ),
        "team_1_avg_run_rate": (
            sum(s1["run_rates"]) / len(s1["run_rates"])
            if s1["run_rates"] else 0
        ),
        "team_2_avg_run_rate": (
            sum(s2["run_rates"]) / len(s2["run_rates"])
            if s2["run_rates"] else 0
        )
    })

    # Update AFTER feature calculation
    if pd.notna(row["team_1_runs"]):
        s1["runs"].append(row["team_1_runs"])
        s1["wickets"].append(row["team_1_wickets"])
        s1["run_rates"].append(row["team_1_run_rate"])

    if pd.notna(row["team_2_runs"]):
        s2["runs"].append(row["team_2_runs"])
        s2["wickets"].append(row["team_2_wickets"])
        s2["run_rates"].append(row["team_2_run_rate"])

    s1["matches"] += 1
    s2["matches"] += 1

    if row["winner"] == team1:
        s1["wins"] += 1
    elif row["winner"] == team2:
        s2["wins"] += 1

historical_df = pd.DataFrame(historical_rows)

test_matches = pd.concat(
    [test_matches.reset_index(drop=True), historical_df],
    axis=1
)

print("Historical features added.")

Historical features added.


In [ ]:
test_matches["win_rate_diff"] = (
    test_matches["team_1_win_rate"]
    - test_matches["team_2_win_rate"]
)

test_matches["avg_runs_diff"] = (
    test_matches["team_1_avg_runs"]
    - test_matches["team_2_avg_runs"]
)

test_matches["avg_wickets_diff"] = (
    test_matches["team_1_avg_wickets"]
    - test_matches["team_2_avg_wickets"]
)

test_matches["run_rate_diff"] = (
    test_matches["team_1_avg_run_rate"]
    - test_matches["team_2_avg_run_rate"]
)

print("Difference features added.")

Difference features added.


In [ ]:
recent_results = defaultdict(list)

recent_form_rows = []

for _, row in test_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    form1 = recent_results[team1][-5:]
    form2 = recent_results[team2][-5:]

    recent1 = (
        sum(form1) / len(form1)
        if form1 else 0.5
    )

    recent2 = (
        sum(form2) / len(form2)
        if form2 else 0.5
    )

    recent_form_rows.append({
        "team_1_recent_form": recent1,
        "team_2_recent_form": recent2
    })

    result1 = 1 if row["winner"] == team1 else 0
    result2 = 1 if row["winner"] == team2 else 0

    recent_results[team1].append(result1)
    recent_results[team2].append(result2)

recent_df = pd.DataFrame(recent_form_rows)

test_matches = pd.concat(
    [test_matches.reset_index(drop=True), recent_df],
    axis=1
)

test_matches["recent_form_diff"] = (
    test_matches["team_1_recent_form"]
    - test_matches["team_2_recent_form"]
)

print("Recent form added.")

Recent form added.


In [ ]:
h2h = defaultdict(lambda: {
    "matches": 0,
    "wins": defaultdict(int)
})

h2h_rows = []

for _, row in test_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    key = tuple(sorted([team1, team2]))
    record = h2h[key]

    if record["matches"] == 0:
        h2h_rate = 0.5
    else:
        h2h_rate = (
            record["wins"][team1]
            / record["matches"]
        )

    h2h_rows.append({
        "team_1_h2h_win_rate": h2h_rate
    })

    record["matches"] += 1
    record["wins"][row["winner"]] += 1

h2h_df = pd.DataFrame(h2h_rows)

test_matches = pd.concat(
    [test_matches.reset_index(drop=True), h2h_df],
    axis=1
)

print("H2H feature added.")

H2H feature added.


In [ ]:
elo = defaultdict(lambda: 1500)

elo_rows = []

K = 20

for _, row in test_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    elo1 = elo[team1]
    elo2 = elo[team2]

    elo_rows.append({
        "team_1_elo": elo1,
        "team_2_elo": elo2,
        "elo_diff": elo1 - elo2
    })

    expected1 = 1 / (
        1 + 10 ** ((elo2 - elo1) / 400)
    )

    actual1 = 1 if row["winner"] == team1 else 0

    elo[team1] = elo1 + K * (
        actual1 - expected1
    )

    elo[team2] = elo2 + K * (
        (1 - actual1) - (1 - expected1)
    )

elo_df = pd.DataFrame(elo_rows)

test_matches = pd.concat(
    [test_matches.reset_index(drop=True), elo_df],
    axis=1
)

print("Elo ratings added.")
print("Teams tracked:", len(elo))

Elo ratings added.
Teams tracked: 12


In [ ]:
test_features = [
    "elo_diff",
    "win_rate_diff",
    "recent_form_diff",
    "avg_runs_diff",
    "avg_wickets_diff",
    "run_rate_diff",
    "team_1_h2h_win_rate"
]

print("===== TEST MODEL FEATURES =====")

for i, feature in enumerate(test_features, 1):
    print(f"{i}. {feature}")

print("\nMissing values:")
print(
    test_matches[test_features]
    .isna()
    .sum()
)

print("\nDataset shape:", test_matches.shape)

===== TEST MODEL FEATURES =====
1. elo_diff
2. win_rate_diff
3. recent_form_diff
4. avg_runs_diff
5. avg_wickets_diff
6. run_rate_diff
7. team_1_h2h_win_rate

Missing values:
elo_diff               0
win_rate_diff          0
recent_form_diff       0
avg_runs_diff          0
avg_wickets_diff       0
run_rate_diff          0
team_1_h2h_win_rate    0
dtype: int64

Dataset shape: (2562, 41)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

test_ml = test_matches.dropna(
    subset=test_features + ["target"]
).copy()

X = test_ml[test_features]
y = test_ml["target"]

print("Usable Test matches:", len(test_ml))
print("Feature shape:", X.shape)

print("\nTarget distribution:")
print(y.value_counts())

Usable Test matches: 2562
Feature shape: (2562, 7)

Target distribution:
target
1    1496
0    1066
Name: count, dtype: int64


In [ ]:
split1 = int(len(test_ml) * 0.70)
split2 = int(len(test_ml) * 0.85)

X_train = X.iloc[:split1]
y_train = y.iloc[:split1]

X_val = X.iloc[split1:split2]
y_val = y.iloc[split1:split2]

X_test = X.iloc[split2:]
y_test = y.iloc[split2:]

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

print("\nDate ranges:")
print(
    "Train:",
    test_ml.iloc[0]["date"],
    "→",
    test_ml.iloc[split1 - 1]["date"]
)

print(
    "Validation:",
    test_ml.iloc[split1]["date"],
    "→",
    test_ml.iloc[split2 - 1]["date"]
)

print(
    "Test:",
    test_ml.iloc[split2]["date"],
    "→",
    test_ml.iloc[-1]["date"]
)

Training: 1793
Validation: 384
Testing: 385

Date ranges:
Train: 2002-12-26 00:00:00 → 2020-01-16 00:00:00
Validation: 2020-01-16 00:00:00 → 2023-11-28 00:00:00
Test: 2023-11-28 00:00:00 → 2026-08-27 00:00:00


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

lr_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [ ]:
lr_prob = lr_model.predict_proba(X_test)[:, 1]
rf_prob = rf_model.predict_proba(X_test)[:, 1]

lr_pred = (lr_prob >= 0.50).astype(int)
rf_pred = (rf_prob >= 0.50).astype(int)

print("===== LOGISTIC REGRESSION =====")
print(
    "Accuracy:",
    round(accuracy_score(y_test, lr_pred), 4)
)
print(
    "F1 Score:",
    round(f1_score(y_test, lr_pred), 4)
)
print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, lr_prob), 4)
)

print("\n===== RANDOM FOREST =====")
print(
    "Accuracy:",
    round(accuracy_score(y_test, rf_pred), 4)
)
print(
    "F1 Score:",
    round(f1_score(y_test, rf_pred), 4)
)
print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, rf_prob), 4)
)

===== LOGISTIC REGRESSION =====
Accuracy: 0.7325
F1 Score: 0.7841
ROC-AUC: 0.8155

===== RANDOM FOREST =====
Accuracy: 0.7247
F1 Score: 0.7969
ROC-AUC: 0.7738


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "F1 Score": [
        f1_score(y_test, lr_pred),
        f1_score(y_test, rf_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, lr_prob),
        roc_auc_score(y_test, rf_prob)
    ]
})

comparison = comparison.round(4)

print(comparison)

                 Model  Accuracy  F1 Score  ROC-AUC
0  Logistic Regression    0.7325    0.7841   0.8155
1        Random Forest    0.7247    0.7969   0.7738


In [ ]:
import numpy as np

lr_val_prob = lr_model.predict_proba(X_val)[:, 1]
rf_val_prob = rf_model.predict_proba(X_val)[:, 1]

print("Validation probabilities generated.")
print("LR sample:", lr_val_prob[:5])
print("RF sample:", rf_val_prob[:5])

Validation probabilities generated.
LR sample: [0.26443697 0.39329644 0.26836156 0.21747272 0.17221408]
RF sample: [0.33539391 0.17406204 0.03607103 0.03165711 0.03338879]


In [ ]:
thresholds = np.arange(0.40, 0.71, 0.01)

threshold_results = []

for threshold in thresholds:

    val_pred = (
        lr_val_prob >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_val,
        val_pred
    )

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy
    })

threshold_df = pd.DataFrame(
    threshold_results
)

best_row = threshold_df.loc[
    threshold_df["accuracy"].idxmax()
]

best_threshold = float(
    best_row["threshold"]
)

print("Best threshold:", best_threshold)
print(
    "Validation accuracy:",
    round(best_row["accuracy"], 4)
)

Best threshold: 0.6800000000000003
Validation accuracy: 0.8151


In [ ]:
lr_test_pred_tuned = (
    lr_prob >= best_threshold
).astype(int)

final_accuracy = accuracy_score(
    y_test,
    lr_test_pred_tuned
)

final_f1 = f1_score(
    y_test,
    lr_test_pred_tuned
)

final_auc = roc_auc_score(
    y_test,
    lr_prob
)

print("===== FINAL TEST MODEL =====")
print("Threshold:", best_threshold)
print(
    "Accuracy:",
    round(final_accuracy, 4)
)
print(
    "F1 Score:",
    round(final_f1, 4)
)
print(
    "ROC-AUC:",
    round(final_auc, 4)
)

===== FINAL TEST MODEL =====
Threshold: 0.6800000000000003
Accuracy: 0.7143
F1 Score: 0.7368
ROC-AUC: 0.8155


In [ ]:
test_model_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "F1 Score",
        "ROC-AUC",
        "Threshold"
    ],
    "Value": [
        final_accuracy,
        final_f1,
        final_auc,
        best_threshold
    ]
})

test_model_summary["Value"] = (
    test_model_summary["Value"]
    .round(4)
)

print(test_model_summary)

      Metric   Value
0   Accuracy  0.7143
1   F1 Score  0.7368
2    ROC-AUC  0.8155
3  Threshold  0.6800


In [ ]:
lr_coefficients = (
    lr_model
    .named_steps["model"]
    .coef_[0]
)

feature_coefficients = pd.DataFrame({
    "Feature": test_features,
    "Coefficient": lr_coefficients
})

feature_coefficients["Absolute_Impact"] = (
    feature_coefficients["Coefficient"]
    .abs()
)

feature_coefficients = (
    feature_coefficients
    .sort_values(
        "Absolute_Impact",
        ascending=False
    )
)

print(feature_coefficients)

               Feature  Coefficient  Absolute_Impact
2     recent_form_diff     1.017884         1.017884
6  team_1_h2h_win_rate     0.446002         0.446002
1        win_rate_diff     0.280308         0.280308
3        avg_runs_diff     0.176482         0.176482
5        run_rate_diff    -0.163497         0.163497
0             elo_diff     0.101601         0.101601
4     avg_wickets_diff     0.058253         0.058253


In [ ]:
import joblib
import json

joblib.dump(
    lr_model,
    "test_match_outcome_model.pkl"
)

joblib.dump(
    best_threshold,
    "test_prediction_threshold.pkl"
)

joblib.dump(
    test_features,
    "test_model_features.pkl"
)

test_matches.to_csv(
    "test_match_data_for_prediction.csv",
    index=False
)

feature_coefficients.to_csv(
    "test_feature_coefficients.csv",
    index=False
)

model_info = {
    "model_name": "MATCHAI Test Match Outcome Predictor",
    "algorithm": "Logistic Regression",
    "dataset": "Men's Test Matches",
    "features": test_features,
    "threshold": best_threshold,
    "accuracy": float(final_accuracy),
    "f1_score": float(final_f1),
    "roc_auc": float(final_auc),
    "initial_elo": 1500,
    "elo_k_factor": 20,
    "recent_form_matches": 5
}

with open("test_model_info.json", "w") as f:
    json.dump(
        model_info,
        f,
        indent=4
    )

print("All Test model assets saved successfully.")

All Test model assets saved successfully.


In [ ]:
import zipfile
import os

test_files = [
    "test_match_outcome_model.pkl",
    "test_prediction_threshold.pkl",
    "test_model_features.pkl",
    "test_match_data_for_prediction.csv",
    "test_feature_coefficients.csv",
    "test_model_info.json"
]

with zipfile.ZipFile(
    "MATCHAI_TEST_Model.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for file in test_files:
        if os.path.exists(file):
            zipf.write(file)

print("MATCHAI_TEST_Model.zip created successfully.")

for file in test_files:
    print("✓", file)

MATCHAI_TEST_Model.zip created successfully.
✓ test_match_outcome_model.pkl
✓ test_prediction_threshold.pkl
✓ test_model_features.pkl
✓ test_match_data_for_prediction.csv
✓ test_feature_coefficients.csv
✓ test_model_info.json


In [ ]:
from google.colab import files

files.download("MATCHAI_TEST_Model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from collections import defaultdict

test_team_stats = defaultdict(lambda: {
    "matches": 0,
    "wins": 0,
    "runs": [],
    "wickets": [],
    "run_rates": [],
    "results": []
})

for _, row in test_matches.sort_values("date").iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    # Team 1
    test_team_stats[team1]["matches"] += 1
    test_team_stats[team1]["runs"].append(
        row["team_1_runs"]
    )
    test_team_stats[team1]["wickets"].append(
        row["team_1_wickets"]
    )
    test_team_stats[team1]["run_rates"].append(
        row["team_1_run_rate"]
    )

    result1 = 1 if row["winner"] == team1 else 0

    test_team_stats[team1]["wins"] += result1
    test_team_stats[team1]["results"].append(result1)

    # Team 2
    test_team_stats[team2]["matches"] += 1
    test_team_stats[team2]["runs"].append(
        row["team_2_runs"]
    )
    test_team_stats[team2]["wickets"].append(
        row["team_2_wickets"]
    )
    test_team_stats[team2]["run_rates"].append(
        row["team_2_run_rate"]
    )

    result2 = 1 if row["winner"] == team2 else 0

    test_team_stats[team2]["wins"] += result2
    test_team_stats[team2]["results"].append(result2)

test_teams = sorted(test_team_stats.keys())

print("Teams available:", len(test_teams))

print("\nTest Teams:")
for team in test_teams:
    print("-", team)

Teams available: 12

Test Teams:
- Australia
- Bangladesh
- England
- ICC World XI
- India
- Ireland
- New Zealand
- Pakistan
- South Africa
- Sri Lanka
- West Indies
- Zimbabwe


In [ ]:
test_elo = defaultdict(lambda: 1500)

K = 20

for _, row in test_matches.sort_values("date").iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    elo1 = test_elo[team1]
    elo2 = test_elo[team2]

    expected1 = 1 / (
        1 + 10 ** ((elo2 - elo1) / 400)
    )

    actual1 = 1 if row["winner"] == team1 else 0

    test_elo[team1] = elo1 + K * (
        actual1 - expected1
    )

    test_elo[team2] = elo2 + K * (
        (1 - actual1) - (1 - expected1)
    )

print("Final Test Elo ratings calculated.")
print("Teams tracked:", len(test_elo))

Final Test Elo ratings calculated.
Teams tracked: 12


In [ ]:
test_h2h = defaultdict(lambda: {
    "matches": 0,
    "wins": defaultdict(int)
})

for _, row in test_matches.sort_values("date").iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    key = tuple(sorted([team1, team2]))

    test_h2h[key]["matches"] += 1
    test_h2h[key]["wins"][row["winner"]] += 1

print("Test H2H database created.")
print("Team pair records:", len(test_h2h))

Test H2H database created.
Team pair records: 52


In [ ]:
def predict_test_match(team1, team2):

    if team1 not in test_team_stats:
        return f"Team not found: {team1}"

    if team2 not in test_team_stats:
        return f"Team not found: {team2}"

    if team1 == team2:
        return "Please select two different teams."

    s1 = test_team_stats[team1]
    s2 = test_team_stats[team2]

    # Win rate
    win_rate_1 = (
        s1["wins"] / s1["matches"]
        if s1["matches"] > 0 else 0.5
    )

    win_rate_2 = (
        s2["wins"] / s2["matches"]
        if s2["matches"] > 0 else 0.5
    )

    # Recent form — last 5 Tests
    recent1 = s1["results"][-5:]
    recent2 = s2["results"][-5:]

    recent_form_1 = (
        sum(recent1) / len(recent1)
        if recent1 else 0.5
    )

    recent_form_2 = (
        sum(recent2) / len(recent2)
        if recent2 else 0.5
    )

    # Average runs
    avg_runs_1 = (
        sum(s1["runs"]) / len(s1["runs"])
    )

    avg_runs_2 = (
        sum(s2["runs"]) / len(s2["runs"])
    )

    # Average wickets
    avg_wickets_1 = (
        sum(s1["wickets"]) / len(s1["wickets"])
    )

    avg_wickets_2 = (
        sum(s2["wickets"]) / len(s2["wickets"])
    )

    # Average run rate
    avg_rr_1 = (
        sum(s1["run_rates"]) / len(s1["run_rates"])
    )

    avg_rr_2 = (
        sum(s2["run_rates"]) / len(s2["run_rates"])
    )

    # Elo
    elo1 = test_elo[team1]
    elo2 = test_elo[team2]

    # H2H
    key = tuple(sorted([team1, team2]))
    h2h_record = test_h2h.get(key)

    if (
        h2h_record
        and h2h_record["matches"] > 0
    ):
        h2h_1 = (
            h2h_record["wins"][team1]
            / h2h_record["matches"]
        )
    else:
        h2h_1 = 0.5

    # Feature vector
    features = pd.DataFrame([{
        "elo_diff": elo1 - elo2,
        "win_rate_diff": win_rate_1 - win_rate_2,
        "recent_form_diff": (
            recent_form_1 - recent_form_2
        ),
        "avg_runs_diff": (
            avg_runs_1 - avg_runs_2
        ),
        "avg_wickets_diff": (
            avg_wickets_1 - avg_wickets_2
        ),
        "run_rate_diff": (
            avg_rr_1 - avg_rr_2
        ),
        "team_1_h2h_win_rate": h2h_1
    }])

    # Exact training feature order
    features = features[test_features]

    probability_team1 = (
        lr_model.predict_proba(features)[0][1]
    )

    prediction = (
        team1
        if probability_team1 >= best_threshold
        else team2
    )

    return {
        "team_1": team1,
        "team_2": team2,
        "predicted_winner": prediction,
        "team_1_probability": round(
            probability_team1 * 100,
            2
        ),
        "team_2_probability": round(
            (1 - probability_team1) * 100,
            2
        ),
        "threshold": best_threshold,
        "features": features.iloc[0].to_dict()
    }

In [ ]:
def predict_test_match(team1, team2):

    if team1 not in test_team_stats:
        return f"Team not found: {team1}"

    if team2 not in test_team_stats:
        return f"Team not found: {team2}"

    if team1 == team2:
        return "Please select two different teams."

    s1 = test_team_stats[team1]
    s2 = test_team_stats[team2]

    # Win rate
    win_rate_1 = (
        s1["wins"] / s1["matches"]
        if s1["matches"] > 0 else 0.5
    )

    win_rate_2 = (
        s2["wins"] / s2["matches"]
        if s2["matches"] > 0 else 0.5
    )

    # Recent form — last 5 Tests
    recent1 = s1["results"][-5:]
    recent2 = s2["results"][-5:]

    recent_form_1 = (
        sum(recent1) / len(recent1)
        if recent1 else 0.5
    )

    recent_form_2 = (
        sum(recent2) / len(recent2)
        if recent2 else 0.5
    )

    # Average runs
    avg_runs_1 = (
        sum(s1["runs"]) / len(s1["runs"])
    )

    avg_runs_2 = (
        sum(s2["runs"]) / len(s2["runs"])
    )

    # Average wickets
    avg_wickets_1 = (
        sum(s1["wickets"]) / len(s1["wickets"])
    )

    avg_wickets_2 = (
        sum(s2["wickets"]) / len(s2["wickets"])
    )

    # Average run rate
    avg_rr_1 = (
        sum(s1["run_rates"]) / len(s1["run_rates"])
    )

    avg_rr_2 = (
        sum(s2["run_rates"]) / len(s2["run_rates"])
    )

    # Elo
    elo1 = test_elo[team1]
    elo2 = test_elo[team2]

    # H2H
    key = tuple(sorted([team1, team2]))
    h2h_record = test_h2h.get(key)

    if (
        h2h_record
        and h2h_record["matches"] > 0
    ):
        h2h_1 = (
            h2h_record["wins"][team1]
            / h2h_record["matches"]
        )
    else:
        h2h_1 = 0.5

    # Feature vector
    features = pd.DataFrame([{
        "elo_diff": elo1 - elo2,
        "win_rate_diff": win_rate_1 - win_rate_2,
        "recent_form_diff": (
            recent_form_1 - recent_form_2
        ),
        "avg_runs_diff": (
            avg_runs_1 - avg_runs_2
        ),
        "avg_wickets_diff": (
            avg_wickets_1 - avg_wickets_2
        ),
        "run_rate_diff": (
            avg_rr_1 - avg_rr_2
        ),
        "team_1_h2h_win_rate": h2h_1
    }])

    # Exact training feature order
    features = features[test_features]

    probability_team1 = (
        lr_model.predict_proba(features)[0][1]
    )

    prediction = (
        team1
        if probability_team1 >= best_threshold
        else team2
    )

    return {
        "team_1": team1,
        "team_2": team2,
        "predicted_winner": prediction,
        "team_1_probability": round(
            probability_team1 * 100,
            2
        ),
        "team_2_probability": round(
            (1 - probability_team1) * 100,
            2
        ),
        "threshold": best_threshold,
        "features": features.iloc[0].to_dict()
    }

In [ ]:
team1 = "India"
team2 = "Australia"

result = predict_test_match(team1, team2)

print("===== MATCHAI TEST PREDICTION =====")
print("Team 1:", result["team_1"])
print("Team 2:", result["team_2"])
print("Predicted Winner:", result["predicted_winner"])

print(
    result["team_1"],
    "Probability:",
    result["team_1_probability"],
    "%"
)

print(
    result["team_2"],
    "Probability:",
    result["team_2_probability"],
    "%"
)

print("Threshold:", result["threshold"])

===== MATCHAI TEST PREDICTION =====
Team 1: India
Team 2: Australia
Predicted Winner: India
India Probability: 74.7 %
Australia Probability: 25.3 %
Threshold: 0.6800000000000003


In [ ]:
print("===== MODEL DECISION FEATURES =====")

for feature, value in result["features"].items():
    print(f"{feature}: {value:.4f}")

===== MODEL DECISION FEATURES =====
elo_diff: -122.3833
win_rate_diff: -0.0414
recent_form_diff: 0.4000
avg_runs_diff: 0.2472
avg_wickets_diff: 0.3981
run_rate_diff: -0.1578
team_1_h2h_win_rate: 0.5455


In [ ]:
print("===== MATCHAI TEST MODEL =====")
print("Algorithm:", "Logistic Regression")
print("Dataset:", "Men's Test Matches")

print(
    "Accuracy:",
    round(final_accuracy * 100, 2),
    "%"
)

print(
    "F1 Score:",
    round(final_f1 * 100, 2),
    "%"
)

print(
    "ROC-AUC:",
    round(final_auc, 4)
)

print(
    "Prediction Threshold:",
    best_threshold
)

print(
    "Features:",
    len(test_features)
)

===== MATCHAI TEST MODEL =====
Algorithm: Logistic Regression
Dataset: Men's Test Matches
Accuracy: 71.43 %
F1 Score: 73.68 %
ROC-AUC: 0.8155
Prediction Threshold: 0.6800000000000003
Features: 7


In [ ]:
import os

print("===== TEST MODEL FILES =====")

test_files = [
    "test_match_outcome_model.pkl",
    "test_prediction_threshold.pkl",
    "test_model_features.pkl",
    "test_match_data_for_prediction.csv",
    "test_feature_coefficients.csv",
    "test_model_info.json",
    "MATCHAI_TEST_Model.zip"
]

for file in test_files:

    if os.path.exists(file):
        size = os.path.getsize(file) / (1024 * 1024)
        print(f"✓ {file} ({size:.2f} MB)")
    else:
        print(f"✗ Missing: {file}")

===== TEST MODEL FILES =====
✓ test_match_outcome_model.pkl (0.00 MB)
✓ test_prediction_threshold.pkl (0.00 MB)
✓ test_model_features.pkl (0.00 MB)
✓ test_match_data_for_prediction.csv (1.24 MB)
✓ test_feature_coefficients.csv (0.00 MB)
✓ test_model_info.json (0.00 MB)
✓ MATCHAI_TEST_Model.zip (0.40 MB)


In [ ]:
print("""
========================================
        MATCHAI TEST MODEL READY
========================================

Model        : Logistic Regression
Dataset      : Men's Test Matches
Features     : 7
Recent Form  : Last 5 Matches
Elo          : Enabled
H2H          : Enabled
Threshold    : Validation Tuned

Accuracy     : {:.2f}%
F1 Score     : {:.2f}%
ROC-AUC      : {:.4f}

Prediction Engine : READY
========================================
""".format(
    final_accuracy * 100,
    final_f1 * 100,
    final_auc
))


        MATCHAI TEST MODEL READY

Model        : Logistic Regression
Dataset      : Men's Test Matches
Features     : 7
Recent Form  : Last 5 Matches
Elo          : Enabled
H2H          : Enabled
Threshold    : Validation Tuned

Accuracy     : 71.43%
F1 Score     : 73.68%
ROC-AUC      : 0.8155

Prediction Engine : READY



In [ ]:
import os
import json
import pandas as pd
import numpy as np

# Find the extracted Test JSON folder
print("Folders in /content:")
print(os.listdir("/content")[:30])

Folders in /content:
['.config', 'test_data', 'tests_male_json.zip', 'test_match_data_for_prediction.csv', 'MATCHAI_TEST_Model.zip', 'test_prediction_threshold.pkl', 'test_model_info.json', 'test_feature_coefficients.csv', 'test_model_features.pkl', 'test_match_outcome_model.pkl', 'sample_data']


In [ ]:
import os
import json
import pandas as pd
import numpy as np

TEST_DIR = "/content/test_data"

print("Test data folder found:", os.path.exists(TEST_DIR))

json_files = [f for f in os.listdir(TEST_DIR) if f.endswith(".json")]

print("JSON files found:", len(json_files))
print("First 5 files:", json_files[:5])

Test data folder found: True
JSON files found: 894
First 5 files: ['345972.json', '743965.json', '430881.json', '1233963.json', '249215.json']


In [ ]:
test_matches_v2 = []

for filename in json_files:
    filepath = os.path.join(TEST_DIR, filename)

    with open(filepath, "r") as f:
        data = json.load(f)

    info = data["info"]
    teams = info.get("teams", [])

    if len(teams) != 2:
        continue

    team1 = teams[0]
    team2 = teams[1]

    outcome = info.get("outcome", {})

    # Determine result
    if "winner" in outcome:
        result_type = "win"
        winner = outcome["winner"]

    elif outcome.get("result") == "draw":
        result_type = "draw"
        winner = None

    elif outcome.get("result") == "tie":
        result_type = "tie"
        winner = None

    else:
        continue

    dates = info.get("dates", [])
    match_date = dates[0] if dates else None

    toss = info.get("toss", {})

    test_matches_v2.append({
        "match_id": filename.replace(".json", ""),
        "date": match_date,
        "team_1": team1,
        "team_2": team2,
        "winner": winner,
        "result_type": result_type,
        "venue": info.get("venue", "Unknown"),
        "toss_winner": toss.get("winner"),
        "toss_decision": toss.get("decision")
    })

test_matches_v2 = pd.DataFrame(test_matches_v2)

test_matches_v2["date"] = pd.to_datetime(test_matches_v2["date"])

test_matches_v2 = (
    test_matches_v2
    .sort_values("date")
    .reset_index(drop=True)
)

print("Total Test matches:", len(test_matches_v2))
print("\nResult distribution:")
print(test_matches_v2["result_type"].value_counts())

test_matches_v2.head()

Total Test matches: 894

Result distribution:
result_type
win     725
draw    169
Name: count, dtype: int64


,match_id,date,team_1,team_2,winner,result_type,venue,toss_winner,toss_decision
0,63963,2001-12-19,England,India,None,draw,"M Chinnaswamy Stadium, Bangalore",England,bat
1,64022,2002-12-26,South Africa,Pakistan,South Africa,win,"Kingsmead, Durban",Pakistan,field
2,64012,2002-12-26,Australia,England,Australia,win,Melbourne Cricket Ground,Australia,bat
3,64013,2003-01-02,England,Australia,England,win,Sydney Cricket Ground,England,bat
4,64023,2003-01-02,South Africa,Pakistan,South Africa,win,"Newlands, Cape Town",South Africa,bat


In [ ]:
print("TEST OUTCOME CHECK")
print("=" * 40)

print("Wins:", (test_matches_v2["result_type"] == "win").sum())
print("Draws:", (test_matches_v2["result_type"] == "draw").sum())
print("Ties:", (test_matches_v2["result_type"] == "tie").sum())

print("\nTotal:", len(test_matches_v2))

TEST OUTCOME CHECK
Wins: 725
Draws: 169
Ties: 0

Total: 894


In [ ]:
test_matches = test_matches_v2.copy()

def get_test_target(row):
    if row["result_type"] == "draw":
        return "draw"
    elif row["winner"] == row["team_1"]:
        return "team_1_win"
    elif row["winner"] == row["team_2"]:
        return "team_2_win"
    else:
        return None

test_matches["target_label"] = test_matches.apply(
    get_test_target,
    axis=1
)

# Remove anything that could not be classified
test_matches = test_matches.dropna(
    subset=["target_label"]
).reset_index(drop=True)

# Numerical labels
target_mapping = {
    "team_1_win": 0,
    "team_2_win": 1,
    "draw": 2
}

test_matches["target"] = test_matches["target_label"].map(
    target_mapping
)

print("Target distribution:")
print(test_matches["target_label"].value_counts())

print("\nTarget mapping:")
print(target_mapping)

Target distribution:
target_label
team_1_win    420
team_2_win    305
draw          169
Name: count, dtype: int64

Target mapping:
{'team_1_win': 0, 'team_2_win': 1, 'draw': 2}


In [ ]:
# Rebuild Test innings performance from Cricsheet JSON

test_performance_list = []

for filename in json_files:
    filepath = os.path.join(TEST_DIR, filename)

    with open(filepath, "r") as f:
        data = json.load(f)

    match_id = filename.replace(".json", "")

    team_runs = {}
    team_wickets = {}

    innings_list = data.get("innings", [])

    for innings in innings_list:

        batting_team = innings.get("team")

        if not batting_team:
            continue

        runs = 0
        wickets = 0

        for over in innings.get("overs", []):

            for delivery in over.get("deliveries", []):

                delivery_runs = delivery.get("runs", {})
                runs += delivery_runs.get("total", 0)

                for wicket in delivery.get("wickets", []):

                    wicket_kind = wicket.get("kind", "")

                    # Do not count retired hurt as a lost wicket
                    if wicket_kind not in ["retired hurt"]:
                        wickets += 1

        team_runs[batting_team] = team_runs.get(batting_team, 0) + runs
        team_wickets[batting_team] = team_wickets.get(batting_team, 0) + wickets

    teams = data["info"].get("teams", [])

    if len(teams) != 2:
        continue

    team1 = teams[0]
    team2 = teams[1]

    test_performance_list.append({
        "match_id": match_id,

        "team_1_runs": team_runs.get(team1, 0),
        "team_2_runs": team_runs.get(team2, 0),

        "team_1_wickets": team_wickets.get(team1, 0),
        "team_2_wickets": team_wickets.get(team2, 0)
    })


test_performance = pd.DataFrame(test_performance_list)

print("Performance data shape:", test_performance.shape)

print("\nColumns:")
print(test_performance.columns.tolist())

test_performance.head()

Performance data shape: (894, 5)

Columns:
['match_id', 'team_1_runs', 'team_2_runs', 'team_1_wickets', 'team_2_wickets']


,match_id,team_1_runs,team_2_runs,team_1_wickets,team_2_wickets
0,345972,376,638,10,15
1,743965,415,820,20,10
2,430881,838,760,14,7
3,1233963,483,659,20,6
4,249215,485,362,20,20


In [ ]:
test_matches = test_matches.merge(
    test_performance,
    on="match_id",
    how="left"
)

print("Merged dataset shape:", test_matches.shape)

print("\nMissing values:")
print(
    test_matches[
        [
            "team_1_runs",
            "team_2_runs",
            "team_1_wickets",
            "team_2_wickets"
        ]
    ].isna().sum()
)

Merged dataset shape: (894, 15)

Missing values:
team_1_runs       0
team_2_runs       0
team_1_wickets    0
team_2_wickets    0
dtype: int64


In [ ]:
print("========== TEST PRE-MATCH BASE DATA ==========")

print("Total matches:", len(test_matches))

print("\nOutcomes:")
print(test_matches["target_label"].value_counts())

print("\nTeams:",
      len(set(test_matches["team_1"]) |
          set(test_matches["team_2"])))

print("Venues:", test_matches["venue"].nunique())

print("\nToss decisions:")
print(test_matches["toss_decision"].value_counts())

print("\nDate range:")
print(test_matches["date"].min(), "→", test_matches["date"].max())

========== TEST PRE-MATCH BASE DATA ==========
Total matches: 894

Outcomes:
target_label
team_1_win    420
team_2_win    305
draw          169
Name: count, dtype: int64

Teams: 12
Venues: 173

Toss decisions:
toss_decision
bat      611
field    283
Name: count, dtype: int64

Date range:
2001-12-19 00:00:00 → 2026-08-27 00:00:00


In [ ]:
from collections import defaultdict

team_stats = defaultdict(lambda: {
    "matches": 0,
    "wins": 0,
    "runs": 0,
    "wickets": 0
})

historical_rows = []

for _, row in test_matches.sort_values("date").iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    s1 = team_stats[team1]
    s2 = team_stats[team2]

    # Features BEFORE this match
    team1_win_rate = (
        s1["wins"] / s1["matches"]
        if s1["matches"] > 0 else 0.5
    )

    team2_win_rate = (
        s2["wins"] / s2["matches"]
        if s2["matches"] > 0 else 0.5
    )

    team1_avg_runs = (
        s1["runs"] / s1["matches"]
        if s1["matches"] > 0 else 0
    )

    team2_avg_runs = (
        s2["runs"] / s2["matches"]
        if s2["matches"] > 0 else 0
    )

    team1_avg_wickets = (
        s1["wickets"] / s1["matches"]
        if s1["matches"] > 0 else 0
    )

    team2_avg_wickets = (
        s2["wickets"] / s2["matches"]
        if s2["matches"] > 0 else 0
    )

    historical_rows.append({
        "match_id": row["match_id"],
        "team1_win_rate": team1_win_rate,
        "team2_win_rate": team2_win_rate,
        "team1_avg_runs": team1_avg_runs,
        "team2_avg_runs": team2_avg_runs,
        "team1_avg_wickets": team1_avg_wickets,
        "team2_avg_wickets": team2_avg_wickets
    })

    # Update AFTER the match
    r1 = row["target_label"]

    s1["matches"] += 1
    s2["matches"] += 1

    s1["runs"] += row["team_1_runs"]
    s2["runs"] += row["team_2_runs"]

    s1["wickets"] += row["team_1_wickets"]
    s2["wickets"] += row["team_2_wickets"]

    if r1 == "team_1_win":
        s1["wins"] += 1

    elif r1 == "team_2_win":
        s2["wins"] += 1

historical_stats = pd.DataFrame(historical_rows)

test_matches = test_matches.merge(
    historical_stats,
    on="match_id",
    how="left"
)

print("Historical statistics added.")
print(test_matches[
    [
        "match_id",
        "team1_win_rate",
        "team2_win_rate",
        "team1_avg_runs",
        "team2_avg_runs"
    ]
].head())

Historical statistics added.
  match_id  team1_win_rate  team2_win_rate  team1_avg_runs  team2_avg_runs
0    63963             0.5             0.5             0.0             0.0
1    64022             0.5             0.5             0.0             0.0
2    64012             0.5             0.0             0.0           369.0
3    64013             0.0             1.0           513.0           658.0
4    64023             1.0             0.0           413.0           411.0


In [ ]:
test_matches["win_rate_diff"] = (
    test_matches["team1_win_rate"]
    - test_matches["team2_win_rate"]
)

test_matches["avg_runs_diff"] = (
    test_matches["team1_avg_runs"]
    - test_matches["team2_avg_runs"]
)

test_matches["avg_wickets_diff"] = (
    test_matches["team1_avg_wickets"]
    - test_matches["team2_avg_wickets"]
)

# Historical run rate proxy
test_matches["team1_run_rate"] = (
    test_matches["team1_avg_runs"] / 90
)

test_matches["team2_run_rate"] = (
    test_matches["team2_avg_runs"] / 90
)

test_matches["run_rate_diff"] = (
    test_matches["team1_run_rate"]
    - test_matches["team2_run_rate"]
)

print("Baseline difference features created.")

print(
    test_matches[
        [
            "win_rate_diff",
            "avg_runs_diff",
            "avg_wickets_diff",
            "run_rate_diff"
        ]
    ].describe()
)

Baseline difference features created.
       win_rate_diff  avg_runs_diff  avg_wickets_diff  run_rate_diff
count     894.000000     894.000000        894.000000     894.000000
mean        0.008224       0.174986         -0.016650       0.001944
std         0.213639      76.245443          2.432489       0.847172
min        -1.000000    -634.666667        -17.000000      -7.051852
25%        -0.121017     -33.156444         -1.093723      -0.368405
50%         0.028565       2.224145         -0.137853       0.024713
75%         0.146343      36.563868          0.963540       0.406265
max         1.000000     623.500000         15.500000       6.927778


In [ ]:
recent_results = defaultdict(list)

recent_form_rows = []

for _, row in test_matches.sort_values("date").iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    form1 = recent_results[team1][-5:]
    form2 = recent_results[team2][-5:]

    team1_recent_form = (
        np.mean(form1) if len(form1) > 0 else 0.5
    )

    team2_recent_form = (
        np.mean(form2) if len(form2) > 0 else 0.5
    )

    recent_form_rows.append({
        "match_id": row["match_id"],
        "team1_recent_form": team1_recent_form,
        "team2_recent_form": team2_recent_form
    })

    # Result points
    if row["target_label"] == "team_1_win":
        result1 = 1
        result2 = 0

    elif row["target_label"] == "team_2_win":
        result1 = 0
        result2 = 1

    else:  # draw
        result1 = 0.5
        result2 = 0.5

    recent_results[team1].append(result1)
    recent_results[team2].append(result2)

recent_form_df = pd.DataFrame(recent_form_rows)

test_matches = test_matches.merge(
    recent_form_df,
    on="match_id",
    how="left"
)

test_matches["recent_form_diff"] = (
    test_matches["team1_recent_form"]
    - test_matches["team2_recent_form"]
)

print("Recent form created.")
print(
    test_matches[
        [
            "team1_recent_form",
            "team2_recent_form",
            "recent_form_diff"
        ]
    ].head(10)
)

Recent form created.
   team1_recent_form  team2_recent_form  recent_form_diff
0              0.500           0.500000          0.000000
1              0.500           0.500000          0.000000
2              0.500           0.500000          0.000000
3              0.250           1.000000         -0.750000
4              1.000           0.000000          1.000000
5              0.500           0.500000          0.000000
6              0.500           0.333333          0.166667
7              1.000           0.500000          0.500000
8              0.500           0.833333         -0.333333
9              0.625           0.600000          0.025000


In [ ]:
# Recreate the Test working dataset

test_matches = test_matches_v2.copy()

def get_test_target(row):
    if row["result_type"] == "draw":
        return "draw"
    elif row["winner"] == row["team_1"]:
        return "team_1_win"
    elif row["winner"] == row["team_2"]:
        return "team_2_win"
    else:
        return None

test_matches["target_label"] = test_matches.apply(
    get_test_target,
    axis=1
)

test_matches = test_matches.dropna(
    subset=["target_label"]
).reset_index(drop=True)

target_mapping = {
    "team_1_win": 0,
    "team_2_win": 1,
    "draw": 2
}

test_matches["target"] = test_matches["target_label"].map(
    target_mapping
)

print("test_matches created successfully!")
print("Total matches:", len(test_matches))

print("\nOutcomes:")
print(test_matches["target_label"].value_counts())

test_matches created successfully!
Total matches: 894

Outcomes:
target_label
team_1_win    420
team_2_win    305
draw          169
Name: count, dtype: int64


In [ ]:
# Build Test match performance data

test_performance_list = []

for filename in json_files:

    filepath = os.path.join(TEST_DIR, filename)

    with open(filepath, "r") as f:
        data = json.load(f)

    info = data["info"]
    teams = info.get("teams", [])

    if len(teams) != 2:
        continue

    team1 = teams[0]
    team2 = teams[1]

    team_runs = {}
    team_wickets = {}

    for innings in data.get("innings", []):

        batting_team = innings.get("team")

        if batting_team not in teams:
            continue

        runs = 0
        wickets = 0

        for over in innings.get("overs", []):

            for delivery in over.get("deliveries", []):

                runs += delivery.get("runs", {}).get("total", 0)

                for wicket in delivery.get("wickets", []):
                    if wicket.get("kind") != "retired hurt":
                        wickets += 1

        team_runs[batting_team] = (
            team_runs.get(batting_team, 0) + runs
        )

        team_wickets[batting_team] = (
            team_wickets.get(batting_team, 0) + wickets
        )

    test_performance_list.append({
        "match_id": filename.replace(".json", ""),

        "team_1_runs": team_runs.get(team1, 0),
        "team_2_runs": team_runs.get(team2, 0),

        "team_1_wickets": team_wickets.get(team1, 0),
        "team_2_wickets": team_wickets.get(team2, 0)
    })


test_performance = pd.DataFrame(test_performance_list)

print("Performance rows:", len(test_performance))
print("\nColumns:")
print(test_performance.columns.tolist())

test_performance.head()

Performance rows: 894

Columns:
['match_id', 'team_1_runs', 'team_2_runs', 'team_1_wickets', 'team_2_wickets']


,match_id,team_1_runs,team_2_runs,team_1_wickets,team_2_wickets
0,345972,376,638,10,15
1,743965,415,820,20,10
2,430881,838,760,14,7
3,1233963,483,659,20,6
4,249215,485,362,20,20


In [ ]:
# Build Test match performance data

test_performance_list = []

for filename in json_files:

    filepath = os.path.join(TEST_DIR, filename)

    with open(filepath, "r") as f:
        data = json.load(f)

    info = data["info"]
    teams = info.get("teams", [])

    if len(teams) != 2:
        continue

    team1 = teams[0]
    team2 = teams[1]

    team_runs = {}
    team_wickets = {}

    for innings in data.get("innings", []):

        batting_team = innings.get("team")

        if batting_team not in teams:
            continue

        runs = 0
        wickets = 0

        for over in innings.get("overs", []):

            for delivery in over.get("deliveries", []):

                runs += delivery.get("runs", {}).get("total", 0)

                for wicket in delivery.get("wickets", []):
                    if wicket.get("kind") != "retired hurt":
                        wickets += 1

        team_runs[batting_team] = (
            team_runs.get(batting_team, 0) + runs
        )

        team_wickets[batting_team] = (
            team_wickets.get(batting_team, 0) + wickets
        )

    test_performance_list.append({
        "match_id": filename.replace(".json", ""),

        "team_1_runs": team_runs.get(team1, 0),
        "team_2_runs": team_runs.get(team2, 0),

        "team_1_wickets": team_wickets.get(team1, 0),
        "team_2_wickets": team_wickets.get(team2, 0)
    })


test_performance = pd.DataFrame(test_performance_list)

print("Performance rows:", len(test_performance))
print("\nColumns:")
print(test_performance.columns.tolist())

test_performance.head()

Performance rows: 894

Columns:
['match_id', 'team_1_runs', 'team_2_runs', 'team_1_wickets', 'team_2_wickets']


,match_id,team_1_runs,team_2_runs,team_1_wickets,team_2_wickets
0,345972,376,638,10,15
1,743965,415,820,20,10
2,430881,838,760,14,7
3,1233963,483,659,20,6
4,249215,485,362,20,20


In [ ]:
print("========== TEST DATA ==========")

print("Matches:", len(test_matches))

print("\nOutcomes:")
print(test_matches["target_label"].value_counts())

print("\nVenues:", test_matches["venue"].nunique())

print("\nDate range:")
print(test_matches["date"].min(), "to", test_matches["date"].max())

print("\nSample:")
display(
    test_matches[
        [
            "date",
            "team_1",
            "team_2",
            "venue",
            "target_label"
        ]
    ].head(10)
)

========== TEST DATA ==========
Matches: 894

Outcomes:
target_label
team_1_win    420
team_2_win    305
draw          169
Name: count, dtype: int64

Venues: 173

Date range:
2001-12-19 00:00:00 to 2026-08-27 00:00:00

Sample:


,date,team_1,team_2,venue,target_label
0,2001-12-19,England,India,"M Chinnaswamy Stadium, Bangalore",draw
1,2002-12-26,South Africa,Pakistan,"Kingsmead, Durban",team_1_win
2,2002-12-26,Australia,England,Melbourne Cricket Ground,team_1_win
3,2003-01-02,England,Australia,Sydney Cricket Ground,team_1_win
4,2003-01-02,South Africa,Pakistan,"Newlands, Cape Town",team_1_win
5,2003-05-09,Australia,West Indies,"Antigua Recreation Ground, St John's, Antigua",team_2_win
6,2003-07-18,Bangladesh,Australia,"Marrara Cricket Ground, Darwin",team_2_win
7,2003-07-24,South Africa,England,"Edgbaston, Birmingham",draw
8,2003-08-14,England,South Africa,"Trent Bridge, Nottingham",team_1_win
9,2003-08-21,South Africa,England,"Headingley, Leeds",team_1_win


In [ ]:
# ==========================================
# FIX: Make sure performance data exists
# ==========================================

# If performance data doesn't exist, build it
if "test_performance" not in globals():

    print("Building test_performance...")

    test_performance_list = []

    for filename in json_files:

        filepath = os.path.join(TEST_DIR, filename)

        with open(filepath, "r") as f:
            data = json.load(f)

        info = data["info"]
        teams = info.get("teams", [])

        if len(teams) != 2:
            continue

        team1 = teams[0]
        team2 = teams[1]

        team_runs = {}
        team_wickets = {}

        for innings in data.get("innings", []):

            batting_team = innings.get("team")

            if batting_team not in teams:
                continue

            runs = 0
            wickets = 0

            for over in innings.get("overs", []):

                for delivery in over.get("deliveries", []):

                    runs += delivery.get(
                        "runs", {}
                    ).get("total", 0)

                    for wicket in delivery.get(
                        "wickets", []
                    ):
                        if wicket.get("kind") != "retired hurt":
                            wickets += 1

            team_runs[batting_team] = (
                team_runs.get(batting_team, 0) + runs
            )

            team_wickets[batting_team] = (
                team_wickets.get(batting_team, 0) + wickets
            )

        test_performance_list.append({
            "match_id": filename.replace(".json", ""),
            "team_1_runs": team_runs.get(team1, 0),
            "team_2_runs": team_runs.get(team2, 0),
            "team_1_wickets": team_wickets.get(team1, 0),
            "team_2_wickets": team_wickets.get(team2, 0)
        })

    test_performance = pd.DataFrame(
        test_performance_list
    )

    print(
        "Performance data created:",
        test_performance.shape
    )

# Check whether columns are already present
required_performance = [
    "team_1_runs",
    "team_2_runs",
    "team_1_wickets",
    "team_2_wickets"
]

missing_columns = [
    col for col in required_performance
    if col not in test_matches.columns
]

# Merge only if necessary
if missing_columns:

    print("Missing columns:", missing_columns)
    print("Merging performance data...")

    # Remove possible duplicate performance columns first
    test_matches = test_matches.drop(
        columns=[
            col for col in required_performance
            if col in test_matches.columns
        ],
        errors="ignore"
    )

    test_matches = test_matches.merge(
        test_performance[
            ["match_id"] + required_performance
        ],
        on="match_id",
        how="left"
    )

# Make sure values are numeric
for col in required_performance:
    test_matches[col] = pd.to_numeric(
        test_matches[col],
        errors="coerce"
    ).fillna(0)

print("\nSUCCESS!")
print("Dataset shape:", test_matches.shape)

print("\nPerformance columns:")
print(test_matches[required_performance].head())

print("\nMissing values:")
print(test_matches[required_performance].isna().sum())

Missing columns: ['team_1_runs', 'team_2_runs', 'team_1_wickets', 'team_2_wickets']
Merging performance data...

SUCCESS!
Dataset shape: (894, 15)

Performance columns:
   team_1_runs  team_2_runs  team_1_wickets  team_2_wickets
0          369          238              10              10
1          413          411              10              20
2          658          657              11              20
3          814          589              19              20
4          620          478               7              20

Missing values:
team_1_runs       0
team_2_runs       0
team_1_wickets    0
team_2_wickets    0
dtype: int64


In [ ]:
from collections import defaultdict

# Sort chronologically
test_matches = test_matches.sort_values("date").reset_index(drop=True)

team_history = defaultdict(lambda: {
    "matches": 0,
    "wins": 0,
    "runs": 0,
    "wickets": 0
})

history_rows = []

for _, row in test_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    s1 = team_history[team1]
    s2 = team_history[team2]

    # Features available BEFORE the current match
    history_rows.append({
        "match_id": row["match_id"],

        "team1_win_rate": (
            s1["wins"] / s1["matches"]
            if s1["matches"] > 0 else 0.5
        ),

        "team2_win_rate": (
            s2["wins"] / s2["matches"]
            if s2["matches"] > 0 else 0.5
        ),

        "team1_avg_runs": (
            s1["runs"] / s1["matches"]
            if s1["matches"] > 0 else 0
        ),

        "team2_avg_runs": (
            s2["runs"] / s2["matches"]
            if s2["matches"] > 0 else 0
        ),

        "team1_avg_wickets": (
            s1["wickets"] / s1["matches"]
            if s1["matches"] > 0 else 0
        ),

        "team2_avg_wickets": (
            s2["wickets"] / s2["matches"]
            if s2["matches"] > 0 else 0
        )
    })

    # Update history AFTER the match
    s1["matches"] += 1
    s2["matches"] += 1

    s1["runs"] += row["team_1_runs"]
    s2["runs"] += row["team_2_runs"]

    s1["wickets"] += row["team_1_wickets"]
    s2["wickets"] += row["team_2_wickets"]

    if row["target_label"] == "team_1_win":
        s1["wins"] += 1

    elif row["target_label"] == "team_2_win":
        s2["wins"] += 1

history_df = pd.DataFrame(history_rows)

test_matches = test_matches.merge(
    history_df,
    on="match_id",
    how="left"
)

print("Historical statistics added.")
print(test_matches.shape)

Historical statistics added.
(894, 21)


In [ ]:
# Compare Team 1 against Team 2

test_matches["win_rate_diff"] = (
    test_matches["team1_win_rate"]
    - test_matches["team2_win_rate"]
)

test_matches["avg_runs_diff"] = (
    test_matches["team1_avg_runs"]
    - test_matches["team2_avg_runs"]
)

test_matches["avg_wickets_diff"] = (
    test_matches["team1_avg_wickets"]
    - test_matches["team2_avg_wickets"]
)

# Run-rate proxy for Test cricket
test_matches["team1_run_rate"] = (
    test_matches["team1_avg_runs"] / 90
)

test_matches["team2_run_rate"] = (
    test_matches["team2_avg_runs"] / 90
)

test_matches["run_rate_diff"] = (
    test_matches["team1_run_rate"]
    - test_matches["team2_run_rate"]
)

print("Difference features created.")

display(
    test_matches[
        [
            "win_rate_diff",
            "avg_runs_diff",
            "avg_wickets_diff",
            "run_rate_diff"
        ]
    ].head()
)

Difference features created.


,win_rate_diff,avg_runs_diff,avg_wickets_diff,run_rate_diff
0,0.0,0.0,0.0,0.000000
1,0.0,0.0,0.0,0.000000
2,0.5,-369.0,-10.0,-4.100000
3,-1.0,-145.0,4.0,-1.611111
4,1.0,2.0,-10.0,0.022222


In [ ]:
recent_results = defaultdict(list)

recent_rows = []

for _, row in test_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    form1 = recent_results[team1][-5:]
    form2 = recent_results[team2][-5:]

    recent_rows.append({
        "match_id": row["match_id"],

        "team1_recent_form": (
            np.mean(form1) if form1 else 0.5
        ),

        "team2_recent_form": (
            np.mean(form2) if form2 else 0.5
        )
    })

    # Update after current match
    if row["target_label"] == "team_1_win":
        result1 = 1
        result2 = 0

    elif row["target_label"] == "team_2_win":
        result1 = 0
        result2 = 1

    else:
        result1 = 0.5
        result2 = 0.5

    recent_results[team1].append(result1)
    recent_results[team2].append(result2)

recent_df = pd.DataFrame(recent_rows)

test_matches = test_matches.merge(
    recent_df,
    on="match_id",
    how="left"
)

test_matches["recent_form_diff"] = (
    test_matches["team1_recent_form"]
    - test_matches["team2_recent_form"]
)

print("Recent form created.")

Recent form created.


In [ ]:
h2h_records = defaultdict(list)

h2h_rows = []

for _, row in test_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    key = tuple(sorted([team1, team2]))

    previous = h2h_records[key]

    if len(previous) == 0:

        h2h_rate = 0.5

    else:

        key0 = key[0]

        points = 0

        for result in previous:

            if result == key0:
                points += 1

            elif result == "draw":
                points += 0.5

        key0_rate = points / len(previous)

        if team1 == key0:
            h2h_rate = key0_rate
        else:
            h2h_rate = 1 - key0_rate

    h2h_rows.append({
        "match_id": row["match_id"],
        "team1_h2h_points_rate": h2h_rate
    })

    # Update AFTER current match
    if row["target_label"] == "team_1_win":
        h2h_records[key].append(team1)

    elif row["target_label"] == "team_2_win":
        h2h_records[key].append(team2)

    else:
        h2h_records[key].append("draw")

h2h_df = pd.DataFrame(h2h_rows)

test_matches = test_matches.merge(
    h2h_df,
    on="match_id",
    how="left"
)

print("H2H feature created.")

H2H feature created.


In [ ]:
elo_ratings = defaultdict(lambda: 1500)

elo_rows = []

K = 20

for _, row in test_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    elo1 = elo_ratings[team1]
    elo2 = elo_ratings[team2]

    # Store BEFORE-match Elo
    elo_rows.append({
        "match_id": row["match_id"],
        "team1_elo": elo1,
        "team2_elo": elo2,
        "elo_diff": elo1 - elo2
    })

    expected1 = 1 / (
        1 + 10 ** ((elo2 - elo1) / 400)
    )

    if row["target_label"] == "team_1_win":
        actual1 = 1

    elif row["target_label"] == "team_2_win":
        actual1 = 0

    else:
        actual1 = 0.5

    # Update AFTER match
    elo_ratings[team1] = (
        elo1 + K * (actual1 - expected1)
    )

    elo_ratings[team2] = (
        elo2 + K * ((1 - actual1) - (1 - expected1))
    )

elo_df = pd.DataFrame(elo_rows)

test_matches = test_matches.merge(
    elo_df,
    on="match_id",
    how="left"
)

print("Elo ratings created.")

Elo ratings created.


In [ ]:
test_features_baseline = [
    "elo_diff",
    "win_rate_diff",
    "recent_form_diff",
    "avg_runs_diff",
    "avg_wickets_diff",
    "run_rate_diff",
    "team1_h2h_points_rate"
]

X_base = test_matches[
    test_features_baseline
].copy()

y_base = test_matches["target"]

print("=" * 55)
print("TEST BASELINE")
print("=" * 55)

print("\nFeatures:")
for feature in test_features_baseline:
    print("✓", feature)

print("\nX shape:", X_base.shape)
print("y shape:", y_base.shape)

print("\nTarget distribution:")
print(test_matches["target_label"].value_counts())

print("\nMissing values:")
print(X_base.isna().sum())

TEST BASELINE

Features:
✓ elo_diff
✓ win_rate_diff
✓ recent_form_diff
✓ avg_runs_diff
✓ avg_wickets_diff
✓ run_rate_diff
✓ team1_h2h_points_rate

X shape: (894, 7)
y shape: (894,)

Target distribution:
target_label
team_1_win    420
team_2_win    305
draw          169
Name: count, dtype: int64

Missing values:
elo_diff                 0
win_rate_diff            0
recent_form_diff         0
avg_runs_diff            0
avg_wickets_diff         0
run_rate_diff            0
team1_h2h_points_rate    0
dtype: int64


In [ ]:
# ==========================================
# CELL 14 — Chronological Split
# ==========================================

# Keep chronological order
test_matches = test_matches.sort_values("date").reset_index(drop=True)

X = test_matches[test_features_baseline].copy()
y = test_matches["target"].copy()

n = len(test_matches)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end].copy()
X_val   = X.iloc[train_end:val_end].copy()
X_test  = X.iloc[val_end:].copy()

y_train = y.iloc[:train_end].copy()
y_val   = y.iloc[train_end:val_end].copy()
y_test  = y.iloc[val_end:].copy()

print("TRAIN:", len(X_train))
print("VALIDATION:", len(X_val))
print("TEST:", len(X_test))

print("\nTarget distribution:")
print("Train:")
print(y_train.value_counts().sort_index())

print("\nValidation:")
print(y_val.value_counts().sort_index())

print("\nTest:")
print(y_test.value_counts().sort_index())

TRAIN: 625
VALIDATION: 134
TEST: 135

Target distribution:
Train:
target
0    297
1    191
2    137
Name: count, dtype: int64

Validation:
target
0    57
1    55
2    22
Name: count, dtype: int64

Test:
target
0    66
1    59
2    10
Name: count, dtype: int64


In [ ]:
# ==========================================
# CELL 15 — Logistic Regression
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            multi_class="auto",
            random_state=42
        )
    )
])

lr_pipeline.fit(X_train, y_train)

lr_val_pred = lr_pipeline.predict(X_val)
lr_val_proba = lr_pipeline.predict_proba(X_val)

lr_test_pred = lr_pipeline.predict(X_test)
lr_test_proba = lr_pipeline.predict_proba(X_test)

print("Logistic Regression trained successfully.")
print("Classes:", lr_pipeline.named_steps["model"].classes_)

Logistic Regression trained successfully.
Classes: [0 1 2]


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [ ]:
# ==========================================
# CELL 16 — Random Forest
# ==========================================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=4,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_val_pred = rf_model.predict(X_val)
rf_val_proba = rf_model.predict_proba(X_val)

rf_test_pred = rf_model.predict(X_test)
rf_test_proba = rf_model.predict_proba(X_test)

print("Random Forest trained successfully.")
print("Classes:", rf_model.classes_)

Random Forest trained successfully.
Classes: [0 1 2]


In [ ]:
# ==========================================
# CELL 17 — Evaluate Baseline Models
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)

def evaluate_model(name, y_true, predictions, probabilities):

    accuracy = accuracy_score(y_true, predictions)

    macro_f1 = f1_score(
        y_true,
        predictions,
        average="macro"
    )

    weighted_f1 = f1_score(
        y_true,
        predictions,
        average="weighted"
    )

    auc = roc_auc_score(
        y_true,
        probabilities,
        multi_class="ovr",
        labels=[0, 1, 2]
    )

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Macro F1    : {macro_f1:.4f}")
    print(f"Weighted F1 : {weighted_f1:.4f}")
    print(f"ROC-AUC     : {auc:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            predictions,
            target_names=[
                "Team 1 Win",
                "Team 2 Win",
                "Draw"
            ],
            zero_division=0
        )
    )

    return {
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "ROC-AUC": auc
    }


lr_baseline_results = evaluate_model(
    "LOGISTIC REGRESSION — TEST BASELINE",
    y_test,
    lr_test_pred,
    lr_test_proba
)

rf_baseline_results = evaluate_model(
    "RANDOM FOREST — TEST BASELINE",
    y_test,
    rf_test_pred,
    rf_test_proba
)


LOGISTIC REGRESSION — TEST BASELINE
Accuracy    : 0.6000
Macro F1    : 0.4050
Weighted F1 : 0.5662
ROC-AUC     : 0.5935

Classification Report:
              precision    recall  f1-score   support

  Team 1 Win       0.57      0.83      0.68        66
  Team 2 Win       0.68      0.44      0.54        59
        Draw       0.00      0.00      0.00        10

    accuracy                           0.60       135
   macro avg       0.42      0.42      0.41       135
weighted avg       0.58      0.60      0.57       135


RANDOM FOREST — TEST BASELINE
Accuracy    : 0.5556
Macro F1    : 0.4066
Weighted F1 : 0.5370
ROC-AUC     : 0.5759

Classification Report:
              precision    recall  f1-score   support

  Team 1 Win       0.59      0.82      0.68        66
  Team 2 Win       0.67      0.34      0.45        59
        Draw       0.08      0.10      0.09        10

    accuracy                           0.56       135
   macro avg       0.44      0.42      0.41       135
weighted 

In [ ]:
# ==========================================
# CELL 18 — Baseline Comparison
# ==========================================

baseline_comparison = pd.DataFrame(
    [
        lr_baseline_results,
        rf_baseline_results
    ],
    index=[
        "Logistic Regression",
        "Random Forest"
    ]
)

print("TEST BASELINE COMPARISON")
print("=" * 60)

display(
    baseline_comparison.round(4)
)

TEST BASELINE COMPARISON


,Accuracy,Macro F1,Weighted F1,ROC-AUC
Logistic Regression,0.6000,0.4050,0.5662,0.5935
Random Forest,0.5556,0.4066,0.5370,0.5759


In [ ]:
from collections import defaultdict

# Store venue-specific history for each team
venue_history = defaultdict(lambda: {
    "matches": 0,
    "wins": 0
})

venue_rows = []

for _, row in test_matches.sort_values("date").iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]
    venue = row["venue"]

    key1 = (team1, venue)
    key2 = (team2, venue)

    s1 = venue_history[key1]
    s2 = venue_history[key2]

    # BEFORE current match
    team1_venue_win_rate = (
        s1["wins"] / s1["matches"]
        if s1["matches"] > 0 else 0.5
    )

    team2_venue_win_rate = (
        s2["wins"] / s2["matches"]
        if s2["matches"] > 0 else 0.5
    )

    venue_rows.append({
        "match_id": row["match_id"],
        "team1_venue_win_rate": team1_venue_win_rate,
        "team2_venue_win_rate": team2_venue_win_rate
    })

    # AFTER current match
    s1["matches"] += 1
    s2["matches"] += 1

    if row["target_label"] == "team_1_win":
        s1["wins"] += 1

    elif row["target_label"] == "team_2_win":
        s2["wins"] += 1


venue_df = pd.DataFrame(venue_rows)

test_matches = test_matches.merge(
    venue_df,
    on="match_id",
    how="left"
)

test_matches["venue_win_rate_diff"] = (
    test_matches["team1_venue_win_rate"]
    - test_matches["team2_venue_win_rate"]
)

print("Venue features created.")

display(
    test_matches[
        [
            "team_1",
            "team_2",
            "venue",
            "team1_venue_win_rate",
            "team2_venue_win_rate",
            "venue_win_rate_diff"
        ]
    ].head(10)
)

Venue features created.


,team_1,team_2,venue,team1_venue_win_rate,team2_venue_win_rate,venue_win_rate_diff
0,England,India,"M Chinnaswamy Stadium, Bangalore",0.5,0.5,0.0
1,South Africa,Pakistan,"Kingsmead, Durban",0.5,0.5,0.0
2,Australia,England,Melbourne Cricket Ground,0.5,0.5,0.0
3,England,Australia,Sydney Cricket Ground,0.5,0.5,0.0
4,South Africa,Pakistan,"Newlands, Cape Town",0.5,0.5,0.0
5,Australia,West Indies,"Antigua Recreation Ground, St John's, Antigua",0.5,0.5,0.0
6,Bangladesh,Australia,"Marrara Cricket Ground, Darwin",0.5,0.5,0.0
7,South Africa,England,"Edgbaston, Birmingham",0.5,0.5,0.0
8,England,South Africa,"Trent Bridge, Nottingham",0.5,0.5,0.0
9,South Africa,England,"Headingley, Leeds",0.5,0.5,0.0


In [ ]:
test_features_venue = test_features_baseline + [
    "venue_win_rate_diff"
]

X_venue = test_matches[
    test_features_venue
].copy()

y_venue = test_matches["target"].copy()

# Same chronological split
n = len(test_matches)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_venue_train = X_venue.iloc[:train_end]
X_venue_val = X_venue.iloc[train_end:val_end]
X_venue_test = X_venue.iloc[val_end:]

y_venue_train = y_venue.iloc[:train_end]
y_venue_val = y_venue.iloc[train_end:val_end]
y_venue_test = y_venue.iloc[val_end:]

print("Features:", len(test_features_venue))
print("Training:", len(X_venue_train))
print("Validation:", len(X_venue_val))
print("Testing:", len(X_venue_test))

print("\nMissing values:")
print(X_venue.isna().sum())

Features: 8
Training: 625
Validation: 134
Testing: 135

Missing values:
elo_diff                 0
win_rate_diff            0
recent_form_diff         0
avg_runs_diff            0
avg_wickets_diff         0
run_rate_diff            0
team1_h2h_points_rate    0
venue_win_rate_diff      0
dtype: int64


In [ ]:
lr_venue = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

lr_venue.fit(
    X_venue_train,
    y_venue_train
)

lr_venue_pred = lr_venue.predict(
    X_venue_test
)

lr_venue_proba = lr_venue.predict_proba(
    X_venue_test
)

print("Logistic Regression + Venue trained.")

Logistic Regression + Venue trained.


In [ ]:
rf_venue = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=4,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_venue.fit(
    X_venue_train,
    y_venue_train
)

rf_venue_pred = rf_venue.predict(
    X_venue_test
)

rf_venue_proba = rf_venue.predict_proba(
    X_venue_test
)

print("Random Forest + Venue trained.")

Random Forest + Venue trained.


In [ ]:
lr_venue_results = evaluate_model(
    "LOGISTIC REGRESSION + VENUE",
    y_venue_test,
    lr_venue_pred,
    lr_venue_proba
)

rf_venue_results = evaluate_model(
    "RANDOM FOREST + VENUE",
    y_venue_test,
    rf_venue_pred,
    rf_venue_proba
)


LOGISTIC REGRESSION + VENUE
Accuracy    : 0.6148
Macro F1    : 0.4177
Weighted F1 : 0.5833
ROC-AUC     : 0.5903

Classification Report:
              precision    recall  f1-score   support

  Team 1 Win       0.59      0.83      0.69        66
  Team 2 Win       0.70      0.47      0.57        59
        Draw       0.00      0.00      0.00        10

    accuracy                           0.61       135
   macro avg       0.43      0.44      0.42       135
weighted avg       0.59      0.61      0.58       135


RANDOM FOREST + VENUE
Accuracy    : 0.5185
Macro F1    : 0.3823
Weighted F1 : 0.5059
ROC-AUC     : 0.5631

Classification Report:
              precision    recall  f1-score   support

  Team 1 Win       0.57      0.76      0.65        66
  Team 2 Win       0.59      0.32      0.42        59
        Draw       0.07      0.10      0.08        10

    accuracy                           0.52       135
   macro avg       0.41      0.39      0.38       135
weighted avg       0.54  

In [ ]:
comparison = pd.DataFrame(
    [
        lr_baseline_results,
        lr_venue_results,
        rf_baseline_results,
        rf_venue_results
    ],
    index=[
        "LR — Baseline",
        "LR — + Venue",
        "RF — Baseline",
        "RF — + Venue"
    ]
)

print("TEST — BASELINE vs VENUE")
print("=" * 70)

display(comparison.round(4))

TEST — BASELINE vs VENUE


,Accuracy,Macro F1,Weighted F1,ROC-AUC
LR — Baseline,0.6000,0.4050,0.5662,0.5935
LR — + Venue,0.6148,0.4177,0.5833,0.5903
RF — Baseline,0.5556,0.4066,0.5370,0.5759
RF — + Venue,0.5185,0.3823,0.5059,0.5631


In [ ]:
# ==========================================
# CELL 25 — CREATE TOSS FEATURES
# ==========================================

# Toss winner from Team 1 perspective
test_matches["toss_winner_diff"] = 0

test_matches.loc[
    test_matches["toss_winner"] == test_matches["team_1"],
    "toss_winner_diff"
] = 1

test_matches.loc[
    test_matches["toss_winner"] == test_matches["team_2"],
    "toss_winner_diff"
] = -1


# Who bats first?
test_matches["bat_first_diff"] = 0

# Team 1 won toss and chose to bat
mask = (
    (test_matches["toss_winner"] == test_matches["team_1"]) &
    (test_matches["toss_decision"] == "bat")
)

test_matches.loc[mask, "bat_first_diff"] = 1


# Team 2 won toss and chose to bat
mask = (
    (test_matches["toss_winner"] == test_matches["team_2"]) &
    (test_matches["toss_decision"] == "bat")
)

test_matches.loc[mask, "bat_first_diff"] = -1


# Team 1 won toss and chose field
mask = (
    (test_matches["toss_winner"] == test_matches["team_1"]) &
    (test_matches["toss_decision"] == "field")
)

test_matches.loc[mask, "bat_first_diff"] = -1


# Team 2 won toss and chose field
mask = (
    (test_matches["toss_winner"] == test_matches["team_2"]) &
    (test_matches["toss_decision"] == "field")
)

test_matches.loc[mask, "bat_first_diff"] = 1


print("Toss features created successfully!")

print("\nColumns:")
print("toss_winner_diff:", "toss_winner_diff" in test_matches.columns)
print("bat_first_diff:", "bat_first_diff" in test_matches.columns)

print("\nDistribution:")
print(test_matches["toss_winner_diff"].value_counts())
print(test_matches["bat_first_diff"].value_counts())

Toss features created successfully!

Columns:
toss_winner_diff: True
bat_first_diff: True

Distribution:
toss_winner_diff
 1    540
-1    354
Name: count, dtype: int64
bat_first_diff
 1    661
-1    233
Name: count, dtype: int64


In [ ]:
# ==========================================
# CELL 26 — BASELINE + TOSS DATASET
# ==========================================

test_features_toss = test_features_baseline + [
    "toss_winner_diff",
    "bat_first_diff"
]

X_toss = test_matches[
    test_features_toss
].copy()

y_toss = test_matches["target"].copy()

# Same chronological split
n = len(test_matches)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_toss_train = X_toss.iloc[:train_end].copy()
X_toss_val = X_toss.iloc[train_end:val_end].copy()
X_toss_test = X_toss.iloc[val_end:].copy()

y_toss_train = y_toss.iloc[:train_end].copy()
y_toss_val = y_toss.iloc[train_end:val_end].copy()
y_toss_test = y_toss.iloc[val_end:].copy()

print("TOSS EXPERIMENT DATASET")
print("=" * 50)

print("Total features:", len(test_features_toss))
print("Training:", len(X_toss_train))
print("Validation:", len(X_toss_val))
print("Testing:", len(X_toss_test))

print("\nFeatures:")
for feature in test_features_toss:
    print("✓", feature)

print("\nMissing values:")
print(X_toss.isna().sum())

TOSS EXPERIMENT DATASET
Total features: 9
Training: 625
Validation: 134
Testing: 135

Features:
✓ elo_diff
✓ win_rate_diff
✓ recent_form_diff
✓ avg_runs_diff
✓ avg_wickets_diff
✓ run_rate_diff
✓ team1_h2h_points_rate
✓ toss_winner_diff
✓ bat_first_diff

Missing values:
elo_diff                 0
win_rate_diff            0
recent_form_diff         0
avg_runs_diff            0
avg_wickets_diff         0
run_rate_diff            0
team1_h2h_points_rate    0
toss_winner_diff         0
bat_first_diff           0
dtype: int64


In [ ]:
# ==========================================
# CELL 27 — LOGISTIC REGRESSION + TOSS
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_toss = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

lr_toss.fit(
    X_toss_train,
    y_toss_train
)

lr_toss_pred = lr_toss.predict(
    X_toss_test
)

lr_toss_proba = lr_toss.predict_proba(
    X_toss_test
)

print("Logistic Regression + Toss trained successfully!")
print("Classes:", lr_toss.named_steps["model"].classes_)

Logistic Regression + Toss trained successfully!
Classes: [0 1 2]


In [ ]:
# ==========================================
# CELL 28 — RANDOM FOREST + TOSS
# ==========================================

from sklearn.ensemble import RandomForestClassifier

rf_toss = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_toss.fit(
    X_toss_train,
    y_toss_train
)

rf_toss_pred = rf_toss.predict(
    X_toss_test
)

rf_toss_proba = rf_toss.predict_proba(
    X_toss_test
)

print("Random Forest + Toss trained successfully!")
print("Classes:", rf_toss.classes_)

Random Forest + Toss trained successfully!
Classes: [0 1 2]


In [ ]:
lr_toss_results = evaluate_model(
    "LOGISTIC REGRESSION + TOSS",
    y_toss_test,
    lr_toss_pred,
    lr_toss_proba
)


LOGISTIC REGRESSION + TOSS
Accuracy    : 0.5778
Macro F1    : 0.3791
Weighted F1 : 0.5318
ROC-AUC     : 0.5896

Classification Report:
              precision    recall  f1-score   support

  Team 1 Win       0.55      0.86      0.67        66
  Team 2 Win       0.68      0.36      0.47        59
        Draw       0.00      0.00      0.00        10

    accuracy                           0.58       135
   macro avg       0.41      0.41      0.38       135
weighted avg       0.56      0.58      0.53       135



In [ ]:
rf_toss_results = evaluate_model(
    "RANDOM FOREST + TOSS",
    y_toss_test,
    rf_toss_pred,
    rf_toss_proba
)


RANDOM FOREST + TOSS
Accuracy    : 0.5333
Macro F1    : 0.3812
Weighted F1 : 0.4979
ROC-AUC     : 0.5870

Classification Report:
              precision    recall  f1-score   support

  Team 1 Win       0.55      0.83      0.66        66
  Team 2 Win       0.64      0.27      0.38        59
        Draw       0.10      0.10      0.10        10

    accuracy                           0.53       135
   macro avg       0.43      0.40      0.38       135
weighted avg       0.56      0.53      0.50       135



In [ ]:
all_test_results = pd.DataFrame(
    [
        lr_baseline_results,
        lr_venue_results,
        lr_toss_results,
        rf_baseline_results,
        rf_venue_results,
        rf_toss_results
    ],
    index=[
        "LR — Baseline",
        "LR — + Venue",
        "LR — + Toss",
        "RF — Baseline",
        "RF — + Venue",
        "RF — + Toss"
    ]
)

print("========== TEST FEATURE EXPERIMENT ==========")

display(
    all_test_results.round(4)
)

========== TEST FEATURE EXPERIMENT ==========


,Accuracy,Macro F1,Weighted F1,ROC-AUC
LR — Baseline,0.6000,0.4050,0.5662,0.5935
LR — + Venue,0.6148,0.4177,0.5833,0.5903
LR — + Toss,0.5778,0.3791,0.5318,0.5896
RF — Baseline,0.5556,0.4066,0.5370,0.5759
RF — + Venue,0.5185,0.3823,0.5059,0.5631
RF — + Toss,0.5333,0.3812,0.4979,0.5870


In [ ]:
print("========== IMPROVEMENT OVER BASELINE ==========")

improvement = all_test_results.copy()

for metric in [
    "Accuracy",
    "Macro F1",
    "Weighted F1",
    "ROC-AUC"
]:

    print(f"\n{metric}")
    print("-" * 40)

    lr_base = lr_baseline_results[metric]
    rf_base = rf_baseline_results[metric]

    print(
        f"LR + Venue : "
        f"{lr_venue_results[metric] - lr_base:+.4f}"
    )

    print(
        f"LR + Toss  : "
        f"{lr_toss_results[metric] - lr_base:+.4f}"
    )

    print(
        f"RF + Venue : "
        f"{rf_venue_results[metric] - rf_base:+.4f}"
    )

    print(
        f"RF + Toss  : "
        f"{rf_toss_results[metric] - rf_base:+.4f}"
    )


========== IMPROVEMENT OVER BASELINE ==========

Accuracy
----------------------------------------
LR + Venue : +0.0148
LR + Toss  : -0.0222
RF + Venue : -0.0370
RF + Toss  : -0.0222

Macro F1
----------------------------------------
LR + Venue : +0.0127
LR + Toss  : -0.0259
RF + Venue : -0.0243
RF + Toss  : -0.0254

Weighted F1
----------------------------------------
LR + Venue : +0.0171
LR + Toss  : -0.0345
RF + Venue : -0.0312
RF + Toss  : -0.0392

ROC-AUC
----------------------------------------
LR + Venue : -0.0032
LR + Toss  : -0.0038
RF + Venue : -0.0128
RF + Toss  : +0.0111


In [ ]:
all_test_results.to_csv(
    "/content/TEST_feature_experiments.csv"
)

print("Saved:")
print("/content/TEST_feature_experiments.csv")

Saved:
/content/TEST_feature_experiments.csv


In [ ]:
# ==========================================
# CELL 34 — CHECK PITCH DATA
# ==========================================

pitch_keywords = [
    "pitch",
    "pitch_type",
    "pitchType",
    "surface",
    "condition"
]

found_pitch_columns = []

for col in test_matches.columns:

    col_lower = str(col).lower()

    for keyword in pitch_keywords:
        if keyword.lower() in col_lower:
            found_pitch_columns.append(col)
            break

print("Possible pitch-related columns:")

if found_pitch_columns:
    for col in found_pitch_columns:
        print("✓", col)
else:
    print("No historical pitch-type column found.")

print("\nCurrent match columns:")
print(test_matches.columns.tolist())

Possible pitch-related columns:
No historical pitch-type column found.

Current match columns:
['match_id', 'date', 'team_1', 'team_2', 'winner', 'result_type', 'venue', 'toss_winner', 'toss_decision', 'target_label', 'target', 'team_1_runs', 'team_2_runs', 'team_1_wickets', 'team_2_wickets', 'team1_win_rate', 'team2_win_rate', 'team1_avg_runs', 'team2_avg_runs', 'team1_avg_wickets', 'team2_avg_wickets', 'win_rate_diff', 'avg_runs_diff', 'avg_wickets_diff', 'team1_run_rate', 'team2_run_rate', 'run_rate_diff', 'team1_recent_form', 'team2_recent_form', 'recent_form_diff', 'team1_h2h_points_rate', 'team1_elo', 'team2_elo', 'elo_diff', 'team1_venue_win_rate', 'team2_venue_win_rate', 'venue_win_rate_diff', 'toss_winner_diff', 'bat_first_diff']


In [ ]:
# ==========================================
# CELL 35 — MATCHAI PITCH OPTIONS
# ==========================================

pitch_types = [
    "Balanced",
    "Batting Friendly",
    "Bowling Friendly",
    "Spin Friendly",
    "Seam Friendly"
]

print("MATCHAI Pitch Type Options")
print("=" * 40)

for i, pitch in enumerate(pitch_types, 1):
    print(f"{i}. {pitch}")

MATCHAI Pitch Type Options
1. Balanced
2. Batting Friendly
3. Bowling Friendly
4. Spin Friendly
5. Seam Friendly


In [ ]:
# ==========================================
# CELL 36 — PITCH CONTEXT
# ==========================================

# Pitch is currently NOT included in model training
# because we do not have reliable historical pitch labels.

test_pitch_context = {
    "feature_name": "pitch_type",
    "status": "context_only",
    "used_for_training": False,
    "reason": "No reliable historical pitch-type labels in current dataset"
}

print("Pitch Type Status")
print("=" * 40)

for key, value in test_pitch_context.items():
    print(f"{key}: {value}")

Pitch Type Status
feature_name: pitch_type
status: context_only
used_for_training: False
reason: No reliable historical pitch-type labels in current dataset


In [ ]:
# ==========================================
# CELL 37 — FEATURE GROUPS
# ==========================================

print("MATCHAI TEST PRE-MATCH FEATURE PLAN")
print("=" * 55)

print("\nBASELINE FEATURES:")
for feature in test_features_baseline:
    print("✓", feature)

print("\nEXPERIMENTAL VENUE:")
print("✓ venue_win_rate_diff")

print("\nEXPERIMENTAL TOSS:")
print("✓ toss_winner_diff")
print("✓ bat_first_diff")

print("\nPITCH:")
print("• Required in UI")
print("• Currently context-only")
print("• NOT used by ML model")

MATCHAI TEST PRE-MATCH FEATURE PLAN

BASELINE FEATURES:
✓ elo_diff
✓ win_rate_diff
✓ recent_form_diff
✓ avg_runs_diff
✓ avg_wickets_diff
✓ run_rate_diff
✓ team1_h2h_points_rate

EXPERIMENTAL VENUE:
✓ venue_win_rate_diff

EXPERIMENTAL TOSS:
✓ toss_winner_diff
✓ bat_first_diff

PITCH:
• Required in UI
• Currently context-only
• NOT used by ML model


In [ ]:
# ==========================================
# CELL 38 — SAVE TEST FEATURE DATA
# ==========================================

test_matches.to_csv(
    "/content/TEST_prematch_feature_data.csv",
    index=False
)

print("Saved successfully:")
print("/content/TEST_prematch_feature_data.csv")

print("\nRows:", len(test_matches))
print("Columns:", len(test_matches.columns))

Saved successfully:
/content/TEST_prematch_feature_data.csv

Rows: 894
Columns: 39


In [ ]:
print("========== CURRENT TEST EXPERIMENTS ==========")

display(
    all_test_results.round(4)
)

========== CURRENT TEST EXPERIMENTS ==========


,Accuracy,Macro F1,Weighted F1,ROC-AUC
LR — Baseline,0.6000,0.4050,0.5662,0.5935
LR — + Venue,0.6148,0.4177,0.5833,0.5903
LR — + Toss,0.5778,0.3791,0.5318,0.5896
RF — Baseline,0.5556,0.4066,0.5370,0.5759
RF — + Venue,0.5185,0.3823,0.5059,0.5631
RF — + Toss,0.5333,0.3812,0.4979,0.5870


In [ ]:
# ==========================================
# CELL 40 — BASELINE + VENUE + TOSS
# ==========================================

test_features_venue_toss = (
    test_features_baseline
    + [
        "venue_win_rate_diff",
        "toss_winner_diff",
        "bat_first_diff"
    ]
)

X_vt = test_matches[
    test_features_venue_toss
].copy()

y_vt = test_matches["target"].copy()

# Chronological split
n = len(test_matches)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_vt_train = X_vt.iloc[:train_end].copy()
X_vt_val = X_vt.iloc[train_end:val_end].copy()
X_vt_test = X_vt.iloc[val_end:].copy()

y_vt_train = y_vt.iloc[:train_end].copy()
y_vt_val = y_vt.iloc[train_end:val_end].copy()
y_vt_test = y_vt.iloc[val_end:].copy()

print("Venue + Toss dataset created.")
print("Features:", len(test_features_venue_toss))
print("Train:", len(X_vt_train))
print("Validation:", len(X_vt_val))
print("Test:", len(X_vt_test))

print("\nMissing values:")
print(X_vt.isna().sum())

Venue + Toss dataset created.
Features: 10
Train: 625
Validation: 134
Test: 135

Missing values:
elo_diff                 0
win_rate_diff            0
recent_form_diff         0
avg_runs_diff            0
avg_wickets_diff         0
run_rate_diff            0
team1_h2h_points_rate    0
venue_win_rate_diff      0
toss_winner_diff         0
bat_first_diff           0
dtype: int64


In [ ]:
lr_venue_toss = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

lr_venue_toss.fit(
    X_vt_train,
    y_vt_train
)

lr_vt_pred = lr_venue_toss.predict(
    X_vt_test
)

lr_vt_proba = lr_venue_toss.predict_proba(
    X_vt_test
)

print("Logistic Regression + Venue + Toss trained.")

Logistic Regression + Venue + Toss trained.


In [ ]:
rf_venue_toss = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_venue_toss.fit(
    X_vt_train,
    y_vt_train
)

rf_vt_pred = rf_venue_toss.predict(
    X_vt_test
)

rf_vt_proba = rf_venue_toss.predict_proba(
    X_vt_test
)

print("Random Forest + Venue + Toss trained.")

Random Forest + Venue + Toss trained.


In [ ]:
lr_vt_results = evaluate_model(
    "LOGISTIC REGRESSION + VENUE + TOSS",
    y_vt_test,
    lr_vt_pred,
    lr_vt_proba
)

rf_vt_results = evaluate_model(
    "RANDOM FOREST + VENUE + TOSS",
    y_vt_test,
    rf_vt_pred,
    rf_vt_proba
)


LOGISTIC REGRESSION + VENUE + TOSS
Accuracy    : 0.5704
Macro F1    : 0.3773
Weighted F1 : 0.5288
ROC-AUC     : 0.5923

Classification Report:
              precision    recall  f1-score   support

  Team 1 Win       0.54      0.83      0.66        66
  Team 2 Win       0.65      0.37      0.47        59
        Draw       0.00      0.00      0.00        10

    accuracy                           0.57       135
   macro avg       0.40      0.40      0.38       135
weighted avg       0.55      0.57      0.53       135


RANDOM FOREST + VENUE + TOSS
Accuracy    : 0.5407
Macro F1    : 0.3888
Weighted F1 : 0.5115
ROC-AUC     : 0.5741

Classification Report:
              precision    recall  f1-score   support

  Team 1 Win       0.56      0.83      0.67        66
  Team 2 Win       0.68      0.29      0.40        59
        Draw       0.08      0.10      0.09        10

    accuracy                           0.54       135
   macro avg       0.44      0.41      0.39       135
weighted av

In [ ]:
final_experiment_results = pd.DataFrame(
    [
        lr_baseline_results,
        lr_venue_results,
        lr_toss_results,
        lr_vt_results,

        rf_baseline_results,
        rf_venue_results,
        rf_toss_results,
        rf_vt_results
    ],
    index=[
        "LR — Baseline",
        "LR — + Venue",
        "LR — + Toss",
        "LR — + Venue + Toss",

        "RF — Baseline",
        "RF — + Venue",
        "RF — + Toss",
        "RF — + Venue + Toss"
    ]
)

print("========== FINAL TEST FEATURE EXPERIMENT ==========")

display(
    final_experiment_results.round(4)
)

========== FINAL TEST FEATURE EXPERIMENT ==========


,Accuracy,Macro F1,Weighted F1,ROC-AUC
LR — Baseline,0.6000,0.4050,0.5662,0.5935
LR — + Venue,0.6148,0.4177,0.5833,0.5903
LR — + Toss,0.5778,0.3791,0.5318,0.5896
LR — + Venue + Toss,0.5704,0.3773,0.5288,0.5923
RF — Baseline,0.5556,0.4066,0.5370,0.5759
RF — + Venue,0.5185,0.3823,0.5059,0.5631
RF — + Toss,0.5333,0.3812,0.4979,0.5870
RF — + Venue + Toss,0.5407,0.3888,0.5115,0.5741


In [ ]:
test_features_final = [
    "elo_diff",
    "win_rate_diff",
    "recent_form_diff",
    "avg_runs_diff",
    "avg_wickets_diff",
    "run_rate_diff",
    "team1_h2h_points_rate",
    "venue_win_rate_diff",
    "toss_winner_diff",
    "bat_first_diff"
]

X_final = test_matches[test_features_final].copy()
y_final = test_matches["target"].copy()

print("Features:", len(test_features_final))
print(test_features_final)

Features: 10
['elo_diff', 'win_rate_diff', 'recent_form_diff', 'avg_runs_diff', 'avg_wickets_diff', 'run_rate_diff', 'team1_h2h_points_rate', 'venue_win_rate_diff', 'toss_winner_diff', 'bat_first_diff']


In [ ]:
# ==========================================
# FINAL TEST MODEL WITH TOSS
# ==========================================

test_final_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=3000,
            random_state=42
        )
    )
])

test_final_model.fit(
    X_final,
    y_final
)

print("FINAL TEST MODEL TRAINED SUCCESSFULLY")
print("Number of features:", len(test_features_final))
print(
    "Model features:",
    test_final_model.feature_names_in_
)

FINAL TEST MODEL TRAINED SUCCESSFULLY
Number of features: 10
Model features: ['elo_diff' 'win_rate_diff' 'recent_form_diff' 'avg_runs_diff'
 'avg_wickets_diff' 'run_rate_diff' 'team1_h2h_points_rate'
 'venue_win_rate_diff' 'toss_winner_diff' 'bat_first_diff']


In [ ]:
# ==========================================
# CELL 47 — FINAL PREDICTION DATABASE
# ==========================================

# Final team statistics
final_team_stats = defaultdict(lambda: {
    "matches": 0,
    "wins": 0,
    "runs": 0,
    "wickets": 0
})

# Final venue statistics
final_venue_stats = defaultdict(lambda: {
    "matches": 0,
    "wins": 0
})

# Final H2H statistics
final_h2h = defaultdict(list)

# Final Elo ratings
final_elo = defaultdict(lambda: 1500)

for _, row in test_matches.sort_values("date").iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]
    venue = row["venue"]

    # -----------------------------
    # TEAM STATS
    # -----------------------------

    final_team_stats[team1]["matches"] += 1
    final_team_stats[team2]["matches"] += 1

    final_team_stats[team1]["runs"] += row["team_1_runs"]
    final_team_stats[team2]["runs"] += row["team_2_runs"]

    final_team_stats[team1]["wickets"] += row["team_1_wickets"]
    final_team_stats[team2]["wickets"] += row["team_2_wickets"]

    if row["target_label"] == "team_1_win":
        final_team_stats[team1]["wins"] += 1
    elif row["target_label"] == "team_2_win":
        final_team_stats[team2]["wins"] += 1

    # -----------------------------
    # VENUE STATS
    # -----------------------------

    v1 = (team1, venue)
    v2 = (team2, venue)

    final_venue_stats[v1]["matches"] += 1
    final_venue_stats[v2]["matches"] += 1

    if row["target_label"] == "team_1_win":
        final_venue_stats[v1]["wins"] += 1
    elif row["target_label"] == "team_2_win":
        final_venue_stats[v2]["wins"] += 1

    # -----------------------------
    # H2H
    # -----------------------------

    key = tuple(sorted([team1, team2]))

    if row["target_label"] == "team_1_win":
        final_h2h[key].append(team1)
    elif row["target_label"] == "team_2_win":
        final_h2h[key].append(team2)
    else:
        final_h2h[key].append("draw")

    # -----------------------------
    # ELO
    # -----------------------------

    elo1 = final_elo[team1]
    elo2 = final_elo[team2]

    expected1 = 1 / (
        1 + 10 ** ((elo2 - elo1) / 400)
    )

    if row["target_label"] == "team_1_win":
        actual1 = 1
    elif row["target_label"] == "team_2_win":
        actual1 = 0
    else:
        actual1 = 0.5

    K = 20

    final_elo[team1] = (
        elo1 + K * (actual1 - expected1)
    )

    final_elo[team2] = (
        elo2 + K * ((1 - actual1) - (1 - expected1))
    )

print("Prediction database created.")

print("Teams:", len(final_team_stats))
print("Venues:", len(set(v for t, v in final_venue_stats.keys())))
print("H2H pairs:", len(final_h2h))

Prediction database created.
Teams: 12
Venues: 173
H2H pairs: 52


In [ ]:
# ==========================================
# CELL 48 — TEST PREDICTION FUNCTION
# ==========================================

def predict_test_match(
    team1,
    team2,
    venue,
    pitch_type=None
):

    # -----------------------------
    # TEAM HISTORY
    # -----------------------------

    s1 = final_team_stats[team1]
    s2 = final_team_stats[team2]

    team1_win_rate = (
        s1["wins"] / s1["matches"]
        if s1["matches"] > 0 else 0.5
    )

    team2_win_rate = (
        s2["wins"] / s2["matches"]
        if s2["matches"] > 0 else 0.5
    )

    team1_avg_runs = (
        s1["runs"] / s1["matches"]
        if s1["matches"] > 0 else 0
    )

    team2_avg_runs = (
        s2["runs"] / s2["matches"]
        if s2["matches"] > 0 else 0
    )

    team1_avg_wickets = (
        s1["wickets"] / s1["matches"]
        if s1["matches"] > 0 else 0
    )

    team2_avg_wickets = (
        s2["wickets"] / s2["matches"]
        if s2["matches"] > 0 else 0
    )

    # -----------------------------
    # RECENT FORM
    # -----------------------------

    recent1 = []
    recent2 = []

    for _, row in test_matches.sort_values("date").iterrows():

        if row["team_1"] == team1:
            if row["target_label"] == "team_1_win":
                recent1.append(1)
            elif row["target_label"] == "team_2_win":
                recent1.append(0)
            else:
                recent1.append(0.5)

        elif row["team_2"] == team1:
            if row["target_label"] == "team_2_win":
                recent1.append(1)
            elif row["target_label"] == "team_1_win":
                recent1.append(0)
            else:
                recent1.append(0.5)

        if row["team_1"] == team2:
            if row["target_label"] == "team_1_win":
                recent2.append(1)
            elif row["target_label"] == "team_2_win":
                recent2.append(0)
            else:
                recent2.append(0.5)

        elif row["team_2"] == team2:
            if row["target_label"] == "team_2_win":
                recent2.append(1)
            elif row["target_label"] == "team_1_win":
                recent2.append(0)
            else:
                recent2.append(0.5)

    team1_recent_form = (
        np.mean(recent1[-5:])
        if recent1 else 0.5
    )

    team2_recent_form = (
        np.mean(recent2[-5:])
        if recent2 else 0.5
    )

    # -----------------------------
    # H2H
    # -----------------------------

    key = tuple(sorted([team1, team2]))
    previous = final_h2h[key]

    if not previous:

        h2h_rate = 0.5

    else:

        points = 0

        for result in previous:

            if result == team1:
                points += 1
            elif result == "draw":
                points += 0.5

        h2h_rate = points / len(previous)

    # -----------------------------
    # ELO
    # -----------------------------

    elo1 = final_elo[team1]
    elo2 = final_elo[team2]

    # -----------------------------
    # VENUE
    # -----------------------------

    venue1 = final_venue_stats[(team1, venue)]
    venue2 = final_venue_stats[(team2, venue)]

    team1_venue_rate = (
        venue1["wins"] / venue1["matches"]
        if venue1["matches"] > 0 else 0.5
    )

    team2_venue_rate = (
        venue2["wins"] / venue2["matches"]
        if venue2["matches"] > 0 else 0.5
    )

    # -----------------------------
    # FEATURES
    # -----------------------------

    features = pd.DataFrame([{
        "elo_diff": elo1 - elo2,

        "win_rate_diff":
            team1_win_rate - team2_win_rate,

        "recent_form_diff":
            team1_recent_form - team2_recent_form,

        "avg_runs_diff":
            team1_avg_runs - team2_avg_runs,

        "avg_wickets_diff":
            team1_avg_wickets - team2_avg_wickets,

        "run_rate_diff":
            (team1_avg_runs / 90)
            - (team2_avg_runs / 90),

        "team1_h2h_points_rate":
            h2h_rate,

        "venue_win_rate_diff":
            team1_venue_rate - team2_venue_rate
    }])

    # Ensure exact feature order
    features = features[test_features_final]

    probabilities = test_final_model.predict_proba(features)[0]

    class_probabilities = dict(
        zip(
            test_final_model.named_steps["model"].classes_,
            probabilities
        )
    )

    predicted_class = test_final_model.predict(features)[0]

    labels = {
        0: "Team 1 Win",
        1: "Team 2 Win",
        2: "Draw"
    }

    return {
        "team_1": team1,
        "team_2": team2,
        "venue": venue,
        "pitch_type": pitch_type,
        "prediction": labels[predicted_class],
        "team_1_win_probability": round(
            class_probabilities.get(0, 0) * 100, 2
        ),
        "team_2_win_probability": round(
            class_probabilities.get(1, 0) * 100, 2
        ),
        "draw_probability": round(
            class_probabilities.get(2, 0) * 100, 2
        )
    }

In [ ]:
# ==========================================
# CELL 49 — TEST PREDICTION ENGINE
# ==========================================

print("Available teams:")
print(sorted(final_team_stats.keys()))

print("\nAvailable venues:")
print(
    sorted(
        set(v for t, v in final_venue_stats.keys())
    )[:30]
)

Available teams:
['Australia', 'Bangladesh', 'England', 'ICC World XI', 'India', 'Ireland', 'New Zealand', 'Pakistan', 'South Africa', 'Sri Lanka', 'West Indies', 'Zimbabwe']

Available venues:
['AMI Stadium', 'Adelaide Oval', 'Antigua Recreation Ground', "Antigua Recreation Ground, St John's", "Antigua Recreation Ground, St John's, Antigua", 'Arnos Vale Ground, Kingstown', 'Arun Jaitley Stadium', 'Arun Jaitley Stadium, Delhi', 'Asgiriya Stadium', 'Bangabandhu National Stadium', 'Bangabandhu National Stadium, Dhaka', 'Barsapara Cricket Stadium, Guwahati', 'Basin Reserve', 'Basin Reserve, Wellington', 'Bay Oval', 'Bay Oval, Mount Maunganui', 'Beausejour Stadium, Gros Islet', 'Bellerive Oval', 'Bellerive Oval, Hobart', 'Brabourne Stadium', 'Brian Lara Stadium, Tarouba, Trinidad', 'Brisbane Cricket Ground', 'Brisbane Cricket Ground, Woolloongabba', 'Brisbane Cricket Ground, Woolloongabba, Brisbane', 'Chittagong Divisional Stadium', 'Civil Service Cricket Club, Stormont, Belfast', 'Daren S

In [ ]:
result = predict_test_match(
    team1="India",
    team2="Australia",
    venue="Melbourne Cricket Ground",
    pitch_type="Balanced"
)

print("========== MATCHAI TEST PREDICTION ==========")

for key, value in result.items():
    print(f"{key}: {value}")

========== MATCHAI TEST PREDICTION ==========
team_1: India
team_2: Australia
venue: Melbourne Cricket Ground
pitch_type: Balanced
prediction: Team 2 Win
team_1_win_probability: 37.61
team_2_win_probability: 40.3
draw_probability: 22.09


In [ ]:
# ==========================================
# CELL 50 — SAVE FINAL TEST MODEL
# ==========================================

import joblib
import json

joblib.dump(
    test_final_model,
    "/content/test_match_outcome_model.pkl"
)

joblib.dump(
    test_features_final,
    "/content/test_model_features.pkl"
)

# Save metadata
test_model_info = {
    "format": "TEST",
    "model": "Logistic Regression",
    "problem_type": "3-class classification",
    "classes": {
        "0": "Team 1 Win",
        "1": "Team 2 Win",
        "2": "Draw"
    },
    "features": test_features_final,
    "pitch_type_used_for_training": False,
    "toss_used_for_training": False,
    "venue_used_for_training": True,
    "historical_matches": len(test_matches),
    "baseline_accuracy": float(lr_baseline_results["Accuracy"]),
    "venue_model_test_accuracy": float(lr_venue_results["Accuracy"])
}

with open(
    "/content/test_model_info.json",
    "w"
) as f:
    json.dump(
        test_model_info,
        f,
        indent=4
    )

print("FINAL TEST MODEL ASSETS SAVED")
print("=" * 50)
print("test_match_outcome_model.pkl")
print("test_model_features.pkl")
print("test_model_info.json")

FINAL TEST MODEL ASSETS SAVED
test_match_outcome_model.pkl
test_model_features.pkl
test_model_info.json


In [ ]:
# ==========================================
# CELL 51 — AVAILABLE TEST OPTIONS
# ==========================================

print("AVAILABLE TEST TEAMS")
print("=" * 50)

for team in sorted(final_team_stats.keys()):
    print(team)

print("\nAVAILABLE VENUES")
print("=" * 50)

all_test_venues = sorted(
    set(v for _, v in final_venue_stats.keys())
)

for venue in all_test_venues[:50]:
    print(venue)

print("\nTotal teams:", len(final_team_stats))
print("Total venues:", len(all_test_venues))

AVAILABLE TEST TEAMS
Australia
Bangladesh
England
ICC World XI
India
Ireland
New Zealand
Pakistan
South Africa
Sri Lanka
West Indies
Zimbabwe

AVAILABLE VENUES
AMI Stadium
Adelaide Oval
Antigua Recreation Ground
Antigua Recreation Ground, St John's
Antigua Recreation Ground, St John's, Antigua
Arnos Vale Ground, Kingstown
Arun Jaitley Stadium
Arun Jaitley Stadium, Delhi
Asgiriya Stadium
Bangabandhu National Stadium
Bangabandhu National Stadium, Dhaka
Barsapara Cricket Stadium, Guwahati
Basin Reserve
Basin Reserve, Wellington
Bay Oval
Bay Oval, Mount Maunganui
Beausejour Stadium, Gros Islet
Bellerive Oval
Bellerive Oval, Hobart
Brabourne Stadium
Brian Lara Stadium, Tarouba, Trinidad
Brisbane Cricket Ground
Brisbane Cricket Ground, Woolloongabba
Brisbane Cricket Ground, Woolloongabba, Brisbane
Chittagong Divisional Stadium
Civil Service Cricket Club, Stormont, Belfast
Daren Sammy National Cricket Stadium, Gros Islet
Daren Sammy National Cricket Stadium, Gros Islet, St Lucia
Daren Sammy N

In [ ]:
# ==========================================
# CELL 52 — TEST MATCH PREDICTION
# ==========================================

result = predict_test_match(
    team1="India",
    team2="Australia",
    venue="Melbourne Cricket Ground",
    pitch_type="Balanced"
)

print("==========================================")
print("        MATCHAI TEST PREDICTION")
print("==========================================")

print("Team 1:", result["team_1"])
print("Team 2:", result["team_2"])
print("Venue:", result["venue"])
print("Pitch Type:", result["pitch_type"])

print("\nPrediction:")
print(result["prediction"])

print("\nProbabilities:")
print("Team 1 Win:", result["team_1_win_probability"], "%")
print("Team 2 Win:", result["team_2_win_probability"], "%")
print("Draw:", result["draw_probability"], "%")

print("\nProbability total:",
      result["team_1_win_probability"]
      + result["team_2_win_probability"]
      + result["draw_probability"],
      "%")

        MATCHAI TEST PREDICTION
Team 1: India
Team 2: Australia
Venue: Melbourne Cricket Ground
Pitch Type: Balanced

Prediction:
Team 2 Win

Probabilities:
Team 1 Win: 37.61 %
Team 2 Win: 40.3 %
Draw: 22.09 %

Probability total: 100.0 %


In [ ]:
# ==========================================
# CELL 53 — VERIFY TEST ENGINE
# ==========================================

required_result_keys = [
    "team_1",
    "team_2",
    "venue",
    "pitch_type",
    "prediction",
    "team_1_win_probability",
    "team_2_win_probability",
    "draw_probability"
]

print("TEST ENGINE VERIFICATION")
print("=" * 50)

all_keys_present = all(
    key in result
    for key in required_result_keys
)

probability_total = (
    result["team_1_win_probability"]
    + result["team_2_win_probability"]
    + result["draw_probability"]
)

valid_prediction = result["prediction"] in [
    "Team 1 Win",
    "Team 2 Win",
    "Draw"
]

print("All result fields:", all_keys_present)
print("Valid prediction:", valid_prediction)
print("Probability total:", probability_total, "%")

if all_keys_present and valid_prediction and probability_total == 100:
    print("\n✓ TEST PREDICTION ENGINE PASSED")
else:
    print("\n⚠ Check prediction output")

TEST ENGINE VERIFICATION
All result fields: True
Valid prediction: True
Probability total: 100.0 %

✓ TEST PREDICTION ENGINE PASSED


In [ ]:
# ==========================================
# CELL 54 — SAVE SUPPORTING DATA
# ==========================================

# Historical match data
test_matches.to_csv(
    "/content/test_match_data_for_prediction.csv",
    index=False
)

# Team statistics
team_stats_rows = []

for team, stats in final_team_stats.items():

    team_stats_rows.append({
        "team": team,
        "matches": stats["matches"],
        "wins": stats["wins"],
        "runs": stats["runs"],
        "wickets": stats["wickets"]
    })

pd.DataFrame(team_stats_rows).to_csv(
    "/content/test_team_stats.csv",
    index=False
)

# Venue statistics
venue_stats_rows = []

for (team, venue), stats in final_venue_stats.items():

    venue_stats_rows.append({
        "team": team,
        "venue": venue,
        "matches": stats["matches"],
        "wins": stats["wins"]
    })

pd.DataFrame(venue_stats_rows).to_csv(
    "/content/test_venue_stats.csv",
    index=False
)

# H2H
h2h_rows = []

for teams, results in final_h2h.items():

    h2h_rows.append({
        "team_pair": " vs ".join(teams),
        "results": "|".join(results)
    })

pd.DataFrame(h2h_rows).to_csv(
    "/content/test_h2h_data.csv",
    index=False
)

# Elo
elo_rows = []

for team, rating in final_elo.items():

    elo_rows.append({
        "team": team,
        "elo": rating
    })

pd.DataFrame(elo_rows).to_csv(
    "/content/test_elo_ratings.csv",
    index=False
)

print("Supporting files saved successfully.")

Supporting files saved successfully.


In [ ]:
# ==========================================
# CELL 55 — CREATE MATCHAI TEST ZIP
# ==========================================

import zipfile
import os

test_files = [
    "/content/test_match_outcome_model.pkl",
    "/content/test_model_features.pkl",
    "/content/test_model_info.json",
    "/content/test_match_data_for_prediction.csv",
    "/content/test_team_stats.csv",
    "/content/test_venue_stats.csv",
    "/content/test_h2h_data.csv",
    "/content/test_elo_ratings.csv"
]

zip_path = "/content/MATCHAI_TEST_Final_Model.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for file_path in test_files:

        if os.path.exists(file_path):

            zipf.write(
                file_path,
                arcname=os.path.basename(file_path)
            )

print("==========================================")
print("MATCHAI TEST PACKAGE CREATED")
print("==========================================")

print("\nZIP:")
print(zip_path)

print("\nFiles included:")

with zipfile.ZipFile(zip_path, "r") as zipf:
    for name in zipf.namelist():
        print("✓", name)

MATCHAI TEST PACKAGE CREATED

ZIP:
/content/MATCHAI_TEST_Final_Model.zip

Files included:
✓ test_match_outcome_model.pkl
✓ test_model_features.pkl
✓ test_model_info.json
✓ test_match_data_for_prediction.csv
✓ test_team_stats.csv
✓ test_venue_stats.csv
✓ test_h2h_data.csv
✓ test_elo_ratings.csv


In [ ]:
# ==========================================
# CELL 56 — DOWNLOAD FINAL TEST MODEL
# ==========================================

from google.colab import files

print("Downloading MATCHAI TEST final model...")

files.download(
    "/content/MATCHAI_TEST_Final_Model.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==========================================
# CELL 45 — FINAL TEST FEATURES WITH TOSS
# ==========================================

test_features_final = [
    "elo_diff",
    "win_rate_diff",
    "recent_form_diff",
    "avg_runs_diff",
    "avg_wickets_diff",
    "run_rate_diff",
    "team1_h2h_points_rate",
    "venue_win_rate_diff",
    "toss_winner_diff",
    "bat_first_diff"
]

X_final = test_matches[test_features_final].copy()
y_final = test_matches["target"].copy()

print("FINAL TEST FEATURES")
print("=" * 55)

for i, feature in enumerate(test_features_final, 1):
    print(f"{i}. {feature}")

print("\nDataset:", X_final.shape)

print("\nMissing values:")
print(X_final.isna().sum())

FINAL TEST FEATURES
1. elo_diff
2. win_rate_diff
3. recent_form_diff
4. avg_runs_diff
5. avg_wickets_diff
6. run_rate_diff
7. team1_h2h_points_rate
8. venue_win_rate_diff
9. toss_winner_diff
10. bat_first_diff

Dataset: (894, 10)

Missing values:
elo_diff                 0
win_rate_diff            0
recent_form_diff         0
avg_runs_diff            0
avg_wickets_diff         0
run_rate_diff            0
team1_h2h_points_rate    0
venue_win_rate_diff      0
toss_winner_diff         0
bat_first_diff           0
dtype: int64


In [ ]:
# ==========================================
# CELL 45 — FINAL TEST FEATURES WITH TOSS
# ==========================================

test_features_final = [
    "elo_diff",
    "win_rate_diff",
    "recent_form_diff",
    "avg_runs_diff",
    "avg_wickets_diff",
    "run_rate_diff",
    "team1_h2h_points_rate",
    "venue_win_rate_diff",
    "toss_winner_diff",
    "bat_first_diff"
]

X_final = test_matches[test_features_final].copy()
y_final = test_matches["target"].copy()

print("FINAL TEST FEATURES")
print("=" * 55)

for i, feature in enumerate(test_features_final, 1):
    print(f"{i}. {feature}")

print("\nDataset:", X_final.shape)

print("\nMissing values:")
print(X_final.isna().sum())

FINAL TEST FEATURES
1. elo_diff
2. win_rate_diff
3. recent_form_diff
4. avg_runs_diff
5. avg_wickets_diff
6. run_rate_diff
7. team1_h2h_points_rate
8. venue_win_rate_diff
9. toss_winner_diff
10. bat_first_diff

Dataset: (894, 10)

Missing values:
elo_diff                 0
win_rate_diff            0
recent_form_diff         0
avg_runs_diff            0
avg_wickets_diff         0
run_rate_diff            0
team1_h2h_points_rate    0
venue_win_rate_diff      0
toss_winner_diff         0
bat_first_diff           0
dtype: int64


In [ ]:
# ==========================================
# CELL 47 — SAVE FINAL TEST MODEL
# ==========================================

import joblib
import json

joblib.dump(
    test_final_model,
    "/content/test_match_outcome_model.pkl"
)

joblib.dump(
    test_features_final,
    "/content/test_model_features.pkl"
)

test_model_info = {
    "format": "TEST",
    "model": "Logistic Regression",
    "problem_type": "3-class classification",

    "classes": {
        "0": "Team 1 Win",
        "1": "Team 2 Win",
        "2": "Draw"
    },

    "features": test_features_final,

    "venue_used": True,
    "toss_used": True,
    "pitch_type_used": False,

    "pitch_type_status": "UI/context only",

    "historical_matches": len(test_matches)
}

with open(
    "/content/test_model_info.json",
    "w"
) as f:
    json.dump(
        test_model_info,
        f,
        indent=4
    )

print("Final Test model saved successfully!")
print("✓ test_match_outcome_model.pkl")
print("✓ test_model_features.pkl")
print("✓ test_model_info.json")

Final Test model saved successfully!
✓ test_match_outcome_model.pkl
✓ test_model_features.pkl
✓ test_model_info.json


In [ ]:
# ==========================================
# CELL 48 — VERIFY FINAL MODEL
# ==========================================

print("FINAL TEST MODEL CHECK")
print("=" * 50)

print("Model loaded:", test_final_model is not None)

print("Number of features:",
      len(test_features_final))

print("Features:")
for feature in test_features_final:
    print("✓", feature)

print("\nClasses:",
      test_final_model.named_steps["model"].classes_)

print("\nModel verification complete.")

FINAL TEST MODEL CHECK
Model loaded: True
Number of features: 10
Features:
✓ elo_diff
✓ win_rate_diff
✓ recent_form_diff
✓ avg_runs_diff
✓ avg_wickets_diff
✓ run_rate_diff
✓ team1_h2h_points_rate
✓ venue_win_rate_diff
✓ toss_winner_diff
✓ bat_first_diff

Classes: [0 1 2]

Model verification complete.


In [ ]:
# ==========================================
# CELL 49 — FINAL TEST PREDICTION FUNCTION
# ==========================================

def predict_test_match(
    team1,
    team2,
    venue,
    pitch_type="Balanced",
    toss_winner=None,
    toss_decision=None
):

    # -----------------------------
    # TEAM STATS
    # -----------------------------

    s1 = final_team_stats[team1]
    s2 = final_team_stats[team2]

    team1_win_rate = (
        s1["wins"] / s1["matches"]
        if s1["matches"] > 0 else 0.5
    )

    team2_win_rate = (
        s2["wins"] / s2["matches"]
        if s2["matches"] > 0 else 0.5
    )

    team1_avg_runs = (
        s1["runs"] / s1["matches"]
        if s1["matches"] > 0 else 0
    )

    team2_avg_runs = (
        s2["runs"] / s2["matches"]
        if s2["matches"] > 0 else 0
    )

    team1_avg_wickets = (
        s1["wickets"] / s1["matches"]
        if s1["matches"] > 0 else 0
    )

    team2_avg_wickets = (
        s2["wickets"] / s2["matches"]
        if s2["matches"] > 0 else 0
    )

    # -----------------------------
    # RECENT FORM
    # -----------------------------

    recent1 = []
    recent2 = []

    for _, row in test_matches.sort_values("date").iterrows():

        if row["team_1"] == team1:
            if row["target_label"] == "team_1_win":
                recent1.append(1)
            elif row["target_label"] == "team_2_win":
                recent1.append(0)
            else:
                recent1.append(0.5)

        elif row["team_2"] == team1:
            if row["target_label"] == "team_2_win":
                recent1.append(1)
            elif row["target_label"] == "team_1_win":
                recent1.append(0)
            else:
                recent1.append(0.5)

        if row["team_1"] == team2:
            if row["target_label"] == "team_1_win":
                recent2.append(1)
            elif row["target_label"] == "team_2_win":
                recent2.append(0)
            else:
                recent2.append(0.5)

        elif row["team_2"] == team2:
            if row["target_label"] == "team_2_win":
                recent2.append(1)
            elif row["target_label"] == "team_1_win":
                recent2.append(0)
            else:
                recent2.append(0.5)

    team1_recent_form = (
        np.mean(recent1[-5:]) if recent1 else 0.5
    )

    team2_recent_form = (
        np.mean(recent2[-5:]) if recent2 else 0.5
    )

    # -----------------------------
    # H2H
    # -----------------------------

    key = tuple(sorted([team1, team2]))
    previous = final_h2h[key]

    if not previous:

        h2h_rate = 0.5

    else:

        points = 0

        for result in previous:

            if result == team1:
                points += 1
            elif result == "draw":
                points += 0.5

        h2h_rate = points / len(previous)

    # -----------------------------
    # ELO
    # -----------------------------

    elo1 = final_elo[team1]
    elo2 = final_elo[team2]

    # -----------------------------
    # VENUE
    # -----------------------------

    venue1 = final_venue_stats[(team1, venue)]
    venue2 = final_venue_stats[(team2, venue)]

    team1_venue_rate = (
        venue1["wins"] / venue1["matches"]
        if venue1["matches"] > 0 else 0.5
    )

    team2_venue_rate = (
        venue2["wins"] / venue2["matches"]
        if venue2["matches"] > 0 else 0.5
    )

    # -----------------------------
    # TOSS
    # -----------------------------

    toss_winner_diff = 0
    bat_first_diff = 0

    if toss_winner is not None:

        if toss_winner == team1:
            toss_winner_diff = 1

        elif toss_winner == team2:
            toss_winner_diff = -1

    if (
        toss_winner == team1
        and toss_decision == "bat"
    ):
        bat_first_diff = 1

    elif (
        toss_winner == team2
        and toss_decision == "bat"
    ):
        bat_first_diff = -1

    elif (
        toss_winner == team1
        and toss_decision == "field"
    ):
        bat_first_diff = -1

    elif (
        toss_winner == team2
        and toss_decision == "field"
    ):
        bat_first_diff = 1

    # -----------------------------
    # FEATURES
    # -----------------------------

    features = pd.DataFrame([{
        "elo_diff": elo1 - elo2,

        "win_rate_diff":
            team1_win_rate - team2_win_rate,

        "recent_form_diff":
            team1_recent_form - team2_recent_form,

        "avg_runs_diff":
            team1_avg_runs - team2_avg_runs,

        "avg_wickets_diff":
            team1_avg_wickets - team2_avg_wickets,

        "run_rate_diff":
            (team1_avg_runs / 90)
            - (team2_avg_runs / 90),

        "team1_h2h_points_rate":
            h2h_rate,

        "venue_win_rate_diff":
            team1_venue_rate - team2_venue_rate,

        "toss_winner_diff":
            toss_winner_diff,

        "bat_first_diff":
            bat_first_diff
    }])

    features = features[test_features_final]

    # -----------------------------
    # PREDICTION
    # -----------------------------

    probabilities = test_final_model.predict_proba(
        features
    )[0]

    classes = test_final_model.named_steps[
        "model"
    ].classes_

    probability_map = dict(
        zip(classes, probabilities)
    )

    predicted_class = test_final_model.predict(
        features
    )[0]

    labels = {
        0: "Team 1 Win",
        1: "Team 2 Win",
        2: "Draw"
    }

    return {
        "team_1": team1,
        "team_2": team2,
        "venue": venue,
        "pitch_type": pitch_type,
        "toss_winner": toss_winner,
        "toss_decision": toss_decision,
        "prediction": labels[predicted_class],

        "team_1_win_probability": round(
            probability_map.get(0, 0) * 100, 2
        ),

        "team_2_win_probability": round(
            probability_map.get(1, 0) * 100, 2
        ),

        "draw_probability": round(
            probability_map.get(2, 0) * 100, 2
        )
    }

print("Final Test prediction function created!")

Final Test prediction function created!


In [ ]:
# ==========================================
# CELL 50 — TEST FINAL PREDICTION
# ==========================================

available_teams = sorted(final_team_stats.keys())
available_venues = sorted(
    set(v for _, v in final_venue_stats.keys())
)

team1 = available_teams[0]
team2 = available_teams[1]

candidate_venues = [
    venue
    for venue in available_venues
    if (team1, venue) in final_venue_stats
    or (team2, venue) in final_venue_stats
]

venue = candidate_venues[0]

print("Using:")
print("Team 1:", team1)
print("Team 2:", team2)
print("Venue:", venue)

result = predict_test_match(
    team1=team1,
    team2=team2,
    venue=venue,
    pitch_type="Balanced",
    toss_winner=team1,
    toss_decision="bat"
)

print("\n==========================================")
print("       MATCHAI TEST PREDICTION")
print("==========================================")

print("Team 1:", result["team_1"])
print("Team 2:", result["team_2"])
print("Venue:", result["venue"])
print("Pitch:", result["pitch_type"])
print("Toss Winner:", result["toss_winner"])
print("Toss Decision:", result["toss_decision"])

print("\nPrediction:", result["prediction"])

print("\nProbabilities:")
print("Team 1 Win:", result["team_1_win_probability"], "%")
print("Team 2 Win:", result["team_2_win_probability"], "%")
print("Draw:", result["draw_probability"], "%")

print(
    "\nTotal:",
    result["team_1_win_probability"]
    + result["team_2_win_probability"]
    + result["draw_probability"],
    "%"
)

Using:
Team 1: Australia
Team 2: Bangladesh
Venue: AMI Stadium

       MATCHAI TEST PREDICTION
Team 1: Australia
Team 2: Bangladesh
Venue: AMI Stadium
Pitch: Balanced
Toss Winner: Australia
Toss Decision: bat

Prediction: Team 1 Win

Probabilities:
Team 1 Win: 85.86 %
Team 2 Win: 6.74 %
Draw: 7.4 %

Total: 100.0 %


In [ ]:
# ==========================================
# CELL 51 — FINAL TEST MODEL VERIFICATION
# ==========================================

print("========== FINAL TEST MODEL ==========")

print("Model: Logistic Regression")
print("Problem: 3-Class Classification")

print("\nClasses:")
print(
    test_final_model
    .named_steps["model"]
    .classes_
)

print("\nFeatures:", len(test_features_final))

for i, feature in enumerate(test_features_final, 1):
    print(f"{i}. {feature}")

print("\n✓ Final model verification complete.")

========== FINAL TEST MODEL ==========
Model: Logistic Regression
Problem: 3-Class Classification

Classes:
[0 1 2]

Features: 10
1. elo_diff
2. win_rate_diff
3. recent_form_diff
4. avg_runs_diff
5. avg_wickets_diff
6. run_rate_diff
7. team1_h2h_points_rate
8. venue_win_rate_diff
9. toss_winner_diff
10. bat_first_diff

✓ Final model verification complete.


In [ ]:
# ==========================================
# CELL 52 — SAVE FINAL MODEL FILES
# ==========================================

import joblib
import json

# Model
joblib.dump(
    test_final_model,
    "/content/test_match_outcome_model.pkl"
)

# Feature list
joblib.dump(
    test_features_final,
    "/content/test_model_features.pkl"
)

# Model information
test_model_info = {
    "format": "TEST",
    "model": "Logistic Regression",
    "classification": "3-class",

    "classes": {
        "0": "Team 1 Win",
        "1": "Team 2 Win",
        "2": "Draw"
    },

    "features": test_features_final,

    "venue_used": True,
    "toss_used": True,
    "pitch_type_used": False,

    "pitch_type_status": "Context/UI only",

    "historical_matches": int(len(test_matches)),

    "notes": [
        "Venue was retained after feature experiment.",
        "Toss was included as requested.",
        "Pitch type is not used for ML because reliable historical pitch labels are unavailable.",
        "Pre-toss predictions use neutral toss values."
    ]
}

with open(
    "/content/test_model_info.json",
    "w"
) as f:
    json.dump(
        test_model_info,
        f,
        indent=4
    )

print("Final model files saved:")
print("✓ test_match_outcome_model.pkl")
print("✓ test_model_features.pkl")
print("✓ test_model_info.json")

Final model files saved:
✓ test_match_outcome_model.pkl
✓ test_model_features.pkl
✓ test_model_info.json


In [ ]:
# ==========================================
# CELL 53 — SAVE PREDICTION DATABASES
# ==========================================

# Historical matches
test_matches.to_csv(
    "/content/test_match_data_for_prediction.csv",
    index=False
)

# Team statistics
team_stats_rows = []

for team, stats in final_team_stats.items():

    team_stats_rows.append({
        "team": team,
        "matches": stats["matches"],
        "wins": stats["wins"],
        "runs": stats["runs"],
        "wickets": stats["wickets"]
    })

pd.DataFrame(team_stats_rows).to_csv(
    "/content/test_team_stats.csv",
    index=False
)

# Venue statistics
venue_stats_rows = []

for (team, venue), stats in final_venue_stats.items():

    venue_stats_rows.append({
        "team": team,
        "venue": venue,
        "matches": stats["matches"],
        "wins": stats["wins"]
    })

pd.DataFrame(venue_stats_rows).to_csv(
    "/content/test_venue_stats.csv",
    index=False
)

# Elo
elo_rows = []

for team, rating in final_elo.items():

    elo_rows.append({
        "team": team,
        "elo": rating
    })

pd.DataFrame(elo_rows).to_csv(
    "/content/test_elo_ratings.csv",
    index=False
)

# H2H
h2h_rows = []

for teams, results in final_h2h.items():

    h2h_rows.append({
        "team_1": teams[0],
        "team_2": teams[1],
        "results": "|".join(results)
    })

pd.DataFrame(h2h_rows).to_csv(
    "/content/test_h2h_data.csv",
    index=False
)

print("Prediction databases saved successfully.")

Prediction databases saved successfully.


In [ ]:
# ==========================================
# CELL 54 — CHECK FINAL FILES
# ==========================================

import os

final_files = [
    "test_match_outcome_model.pkl",
    "test_model_features.pkl",
    "test_model_info.json",
    "test_match_data_for_prediction.csv",
    "test_team_stats.csv",
    "test_venue_stats.csv",
    "test_elo_ratings.csv",
    "test_h2h_data.csv"
]

print("========== FINAL TEST FILE CHECK ==========")

all_present = True

for filename in final_files:

    path = "/content/" + filename

    if os.path.exists(path):

        size = os.path.getsize(path) / 1024

        print(
            f"✓ {filename} "
            f"({size:.1f} KB)"
        )

    else:

        print(f"✗ MISSING: {filename}")
        all_present = False

print()

if all_present:
    print("✓ ALL TEST MODEL FILES ARE READY")
else:
    print("⚠ Some files are missing.")

========== FINAL TEST FILE CHECK ==========
✓ test_match_outcome_model.pkl (2.2 KB)
✓ test_model_features.pkl (0.2 KB)
✓ test_model_info.json (0.9 KB)
✓ test_match_data_for_prediction.csv (387.3 KB)
✓ test_team_stats.csv (0.4 KB)
✓ test_venue_stats.csv (28.8 KB)
✓ test_elo_ratings.csv (0.3 KB)
✓ test_h2h_data.csv (8.7 KB)

✓ ALL TEST MODEL FILES ARE READY


In [ ]:
# ==========================================
# CELL 55 — CREATE FINAL TEST ZIP
# ==========================================

import zipfile
import os

zip_path = "/content/MATCHAI_TEST_Final_Model.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for filename in final_files:

        filepath = "/content/" + filename

        if os.path.exists(filepath):

            zipf.write(
                filepath,
                arcname=filename
            )

print("==========================================")
print("MATCHAI TEST FINAL PACKAGE")
print("==========================================")

print("\nCreated:")
print(zip_path)

print("\nContents:")

with zipfile.ZipFile(zip_path, "r") as zipf:

    for name in zipf.namelist():
        print("✓", name)

MATCHAI TEST FINAL PACKAGE

Created:
/content/MATCHAI_TEST_Final_Model.zip

Contents:
✓ test_match_outcome_model.pkl
✓ test_model_features.pkl
✓ test_model_info.json
✓ test_match_data_for_prediction.csv
✓ test_team_stats.csv
✓ test_venue_stats.csv
✓ test_elo_ratings.csv
✓ test_h2h_data.csv


In [ ]:
# ==========================================
# CELL 56 — DOWNLOAD FINAL TEST MODEL
# ==========================================

from google.colab import files

print("Downloading MATCHAI TEST model...")

files.download(
    "/content/MATCHAI_TEST_Final_Model.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>